In [ ]:
pip install kaggle_environments

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 20.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 168.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.6/175.6 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 32.7

In [ ]:
from kaggle_environments import make
#pip install kaggle-environments

In [ ]:
standard_configuration = {"episodeSteps": 720,"boardSize":10,"startingMoney":3000,"maxMarketOrdersPerTurn":10,"turnsPerDay":24,"shedCapacity":100,"weedSpawnChance":0.005,"townShopUnlockInterval":3,"townShopSellInterval":4,"townCenterSellInterval":24}
env = make("kaggriculture", configuration={"episodeSteps": 200})
#env.run([my_agent, "random"])
#env.render(mode="ipython", width=800, height=800)

In [ ]:
# ============================================================
# CELL 1
# MODELS + ENCODERS + REWARDS + HELPERS
# ============================================================

import math
import random

from collections import deque, Counter
from dataclasses import dataclass

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.distributions import Categorical

from kaggle_environments import make

import os
import json
import copy


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================================
# GAME CONSTANTS
# ============================================================

BOARD_SIZE = 10

# Kaggriculture competition season: 30 days x 24 turns = 720 turns.
DAYS_PER_GAME = 30
TURNS_PER_DAY = 24
TURNS_PER_GAME = (
    DAYS_PER_GAME * TURNS_PER_DAY
)

assert TURNS_PER_GAME == 720

CROPS = [
    "WHEAT",
    "CARROT",
    "TOMATO",
    "STRAWBERRY",
    "MELON",
]

REGIONS = [
    "NW",
    "NE",
    "SW",
    "SE",
]

ROLES = [
    "MAINTAIN",
    "PLANT",
    "HARVEST",
    "CLEAR",
    "LOGISTICS",
]

MAX_HANDS = 4
MAX_WORKERS = MAX_HANDS + 1

NUM_PHASES = 4


# ============================================================
# CURRENT ADVANCED KAGGRICULTURE CROP PARAMETERS
# ============================================================

CROP_SPECS = {

    "WHEAT": {
        "seed": 10,
        "first_yield_day": 2,
        "max_yield_day": 4,
        "interval": 0,
        "max_yield": 6,
        "ongoing": False,
    },

    "CARROT": {
        "seed": 20,
        "first_yield_day": 2,
        "max_yield_day": 3,
        "interval": 0,
        "max_yield": 4,
        "ongoing": False,
    },

    "TOMATO": {
        "seed": 50,
        "first_yield_day": 8,
        "max_yield_day": 8,
        "interval": 1,
        "max_yield": 4,
        "ongoing": True,
    },

    "STRAWBERRY": {
        "seed": 100,
        "first_yield_day": 10,
        "max_yield_day": 10,
        "interval": 2,
        "max_yield": 4,
        "ongoing": True,
    },

    "MELON": {
        "seed": 80,
        "first_yield_day": 10,
        "max_yield_day": 12,
        "interval": 0,
        "max_yield": 6,
        "ongoing": False,
    },
}


# ============================================================
# MARKET PARAMETERS
# Used only for FUTURE price estimation.
# Current price still comes directly from obs["market"].
# ============================================================

MARKET_I0 = 10_000

MARKET_PARAMS = {

    "WHEAT": {
        "base": 25,
        "T": 400,
        "below_func": "sqrt",
        "below_target": .80,
        "above_func": "log",
        "above_target": .20,
    },

    "CARROT": {
        "base": 35,
        "T": 450,
        "below_func": "hinge",
        "below_target": 1.00,
        "above_func": "sqrt",
        "above_target": .70,
    },

    "TOMATO": {
        "base": 60,
        "T": 200,
        "below_func": "hinge",
        "below_target": .40,
        "above_func": "sqrt",
        "above_target": .60,
    },

    "STRAWBERRY": {
        "base": 120,
        "T": 100,
        "below_func": "sqrt",
        "below_target": .70,
        "above_func": "linear",
        "above_target": 1.60,
    },

    "MELON": {
        "base": 250,
        "T": 300,
        "below_func": "log",
        "below_target": .20,
        "above_func": "sq",
        "above_target": 3.60,
    },
}


# ============================================================
# TOWN / SHOP DEMAND
# ============================================================

SHOPS = [
    "BAKERY",
    "PIZZA_SHOP",
    "BRUNCH_SPOT",
    "YARN_STORE",
    "ICE_CREAM_SHOP",
    "PET_CAFE",
    "SMOOTHIE_SHOP",
    "FARMERS_MARKET",
]

SHOP_PRODUCTS = {

    "BAKERY":
        ["EGG", "WHEAT"],

    "PIZZA_SHOP":
        ["MILK", "TOMATO", "WHEAT"],

    "BRUNCH_SPOT":
        ["EGG", "WHEAT", "STRAWBERRY"],

    "YARN_STORE":
        ["WOOL"],

    "ICE_CREAM_SHOP":
        ["STRAWBERRY", "MILK", "WHEAT"],

    "PET_CAFE":
        ["CARROT"],

    "SMOOTHIE_SHOP":
        ["STRAWBERRY", "MILK"],

    "FARMERS_MARKET":
        ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY"],
}


# ============================================================
# FARM GEOMETRY
# ============================================================

REGION_BOUNDS = {

    0: (0, 5, 0, 5),      # NW
    1: (5, 10, 0, 5),     # NE
    2: (0, 5, 5, 10),     # SW
    3: (5, 10, 5, 10),    # SE
}

SHED_ACCESS = [
    (4, 4),
    (5, 4),
    (4, 5),
    (5, 5),
]

LAND_VALUE = {
    "NW": 0,
    "NE": 1000,
    "SW": 2000,
    "SE": 4000,
}


# ============================================================
# WORKER ACTION SPACE
#
# IMPORTANT:
# Advanced Kaggriculture uses PLACE rather than DROP.
# ============================================================

WORKER_ACTIONS = [

    ("PASS", None),

    ("NORTH", None),
    ("SOUTH", None),
    ("EAST", None),
    ("WEST", None),

    ("WATER", None),
    ("HARVEST", None),
    ("DIG", None),

    ("PLANT", "WHEAT"),
    ("PLANT", "CARROT"),
    ("PLANT", "TOMATO"),
    ("PLANT", "STRAWBERRY"),
    ("PLANT", "MELON"),

    ("PLACE", None),
]


# ============================================================
# BASIC HELPERS
# ============================================================

def one_hot(i, n):

    x = np.zeros(
        n,
        dtype=np.float32
    )

    if 0 <= i < n:
        x[i] = 1.0

    return x


def manhattan(a, b):

    return (
        abs(a[0] - b[0])
        + abs(a[1] - b[1])
    )


def quadrant_index(x, y):

    if x < 5 and y < 5:
        return 0

    if x >= 5 and y < 5:
        return 1

    if x < 5 and y >= 5:
        return 2

    return 3


def get_worker_pos(
    farm,
    worker_idx
):

    if worker_idx == 0:
        return tuple(
            farm["farmer"]
        )

    hands = farm.get(
        "hands", []
    )

    if worker_idx - 1 < len(hands):

        return tuple(
            hands[
                worker_idx - 1
            ]
        )

    # Future hand not yet spawned:
    # approximate its spawn at shed.
    return SHED_ACCESS[
        (worker_idx - 1)
        % len(SHED_ACCESS)
    ]


def is_shed_access(
    x,
    y
):

    return (
        (x, y)
        in SHED_ACCESS
    )


def region_is_unlocked(
    farm,
    region_idx
):

    return (
        REGIONS[region_idx]
        in farm.get(
            "unlocked_quadrants",
            []
        )
    )


# ============================================================
# PLAN OBJECT
# ============================================================

@dataclass
class FarmPlan:

    hire_count: int

    buy_land: int

    regions: list

    roles: list

    crops: list

    target_x: list

    target_y: list

    target_values: list


# ============================================================
# LONG-TERM ECONOMIC VALUE MODEL
#
# These are not literal environment fees.
#
# Worker actions cost TIME. We convert that lost time into
# an approximate dollar opportunity cost.
# ============================================================

ACTION_OPPORTUNITY_COST = 2.0

MAINTENANCE_TRAVEL_FACTOR = 0.20

MARKET_DEMAND_CONFIDENCE = 0.25

SELF_MARKET_IMPACT = 0.50

INVALID_PLANT_VALUE = -1_000_000.0


def _market_shape(
    name,
    x,
    T
):

    x = max(
        0.0,
        float(x)
    )

    if name == "linear":
        return x

    if name == "sq":
        return x * x

    if name == "sqrt":
        return math.sqrt(x)

    if name == "log":
        return math.log(
            1.0 + x
        )

    if name == "log10":
        return math.log10(
            1.0 + x
        )

    if name == "hinge":

        if T <= 0:
            return x

        u = x / T

        return (
            u
            + 8.0
            * max(
                0.0,
                u - 1.0
            ) ** 2
        )

    return x


def estimate_market_price(
    crop,
    inventory
):

    p = MARKET_PARAMS[crop]

    base = p["base"]

    T = p["T"]

    inventory = max(
        0,
        float(inventory)
    )

    if inventory < MARKET_I0:

        distance = (
            MARKET_I0
            - inventory
        )

        shape = _market_shape(
            p["below_func"],
            distance,
            T
        )

        denominator = _market_shape(
            p["below_func"],
            T,
            T
        )

        amp = (
            p["below_target"]
            * base
            / max(
                denominator,
                1e-8
            )
        )

        price = (
            base
            + amp * shape
        )

    else:

        distance = (
            inventory
            - MARKET_I0
        )

        shape = _market_shape(
            p["above_func"],
            distance,
            T
        )

        denominator = _market_shape(
            p["above_func"],
            T,
            T
        )

        amp = (
            p["above_target"]
            * base
            / max(
                denominator,
                1e-8
            )
        )

        price = (
            base
            - amp * shape
        )

    return max(
        1.0,
        float(price)
    )


def seed_price(
    obs,
    crop
):
    """
    Advanced Kaggriculture currently has fixed seed costs.

    If a future/custom environment exposes seed_prices,
    this automatically uses them.
    """

    market_seed_prices = (
        obs.get(
            "market", {}
        ).get(
            "seed_prices", {}
        )
    )

    if crop in market_seed_prices:

        return float(
            market_seed_prices[crop]
        )

    return float(
        CROP_SPECS[crop][
            "seed"
        ]
    )


def expected_market_demand_per_day(
    obs,
    crop
):
    """
    Approximate expected town demand for crop per day.

    Used only to forecast future price.
    """

    day = obs["day"]

    # Town center consumes every 12 turns.
    # 24 turns/day -> ~2 demand events/day.
    if day < 10:
        center_scale = 1

    elif day < 20:
        center_scale = 2

    else:
        center_scale = 4

    demand = (
        2.0
        * center_scale
    )

    unlocked = (
        obs.get(
            "town", {}
        ).get(
            "unlocked_shops",
            []
        )
    )

    # Shop interval ~= 4 turns:
    # 24 turns/day -> 6 events/day.
    for shop in unlocked:

        products = (
            SHOP_PRODUCTS.get(
                shop,
                []
            )
        )

        if crop not in products:
            continue

        multiplier = (
            2.0
            if len(products) == 1
            else 1.0
        )

        demand += (
            6.0
            * multiplier
        )

    return demand


def expected_sale_price(
    obs,
    crop,
    days_ahead,
    expected_units=1
):
    """
    Forecast sale price at harvest.

    The prediction is deliberately conservative:
    only a fraction of expected town demand is assumed
    to affect inventory because the opponent may also
    be producing the crop.
    """

    market = obs.get(
        "market", {}
    )

    inventory = (
        market.get(
            "inventory", {}
        ).get(
            crop,
            MARKET_I0
        )
    )

    current_price = (
        market.get(
            "prices", {}
        ).get(
            crop,
            MARKET_PARAMS[crop][
                "base"
            ]
        )
    )

    demand = (
        expected_market_demand_per_day(
            obs,
            crop
        )
        * max(
            0,
            days_ahead
        )
        * MARKET_DEMAND_CONFIDENCE
    )

    own_supply = (
        expected_units
        * SELF_MARKET_IMPACT
    )

    future_inventory = max(
        0,
        inventory
        - demand
        + own_supply
    )

    model_price = (
        estimate_market_price(
            crop,
            future_inventory
        )
    )

    # Blend model forecast with observable current price.
    return (
        0.65
        * model_price
        + 0.35
        * current_price
    )


def expected_crop_yield(
    crop,
    available_days
):
    """
    Returns:
        expected_units,
        days_until_final_harvest
    """

    spec = CROP_SPECS[crop]

    available_days = max(
        0,
        int(available_days)
    )

    first = (
        spec[
            "first_yield_day"
        ]
    )

    if available_days < first:

        return (
            0.0,
            first
        )

    # --------------------------------------------------------
    # ONE-TIME CROP
    # --------------------------------------------------------

    if not spec["ongoing"]:

        harvest_age = min(
            available_days,
            spec[
                "max_yield_day"
            ]
        )

        window_start = (
            spec[
                "max_yield_day"
            ] + 1
        ) // 2

        bonus_days = max(
            0,
            harvest_age
            - window_start
            + 1
        )

        units = min(
            spec["max_yield"],
            1 + bonus_days
        )

        return (
            float(units),
            harvest_age
        )

    # --------------------------------------------------------
    # ONGOING CROP
    # --------------------------------------------------------

    interval = max(
        1,
        spec["interval"]
    )

    production_events = (
        1
        + (
            available_days
            - first
        ) // interval
    )

    units = min(
        spec["max_yield"],
        production_events
    )

    final_harvest_day = (
        first
        + (
            units - 1
        ) * interval
    )

    return (
        float(units),
        int(final_harvest_day)
    )


def long_term_plant_value(
    obs,
    crop,
    x,
    y
):
    """
    Approximate net value of planting crop at (x,y).

    Considers:
      + expected crop revenue
      - seed purchase cost
      - worker travel actions
      - planting action
      - watering / maintenance actions
      - harvest action
      - return-to-shed travel
      - PLACE action

    This is a teacher/heuristic, not the final policy.
    """

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    tile = farm[
        "tiles"
    ][y][x]

    # Must be empty.
    if tile is not None:

        return INVALID_PLANT_VALUE

    region_idx = (
        quadrant_index(
            x,
            y
        )
    )

    if not region_is_unlocked(
        farm,
        region_idx
    ):

        return INVALID_PLANT_VALUE

    days_remaining = (
        DAYS_PER_GAME
        - 1
        - obs["day"]
    )

    expected_units, harvest_days = (
        expected_crop_yield(
            crop,
            days_remaining
        )
    )

    # Crop cannot produce before game ends.
    if expected_units <= 0:

        return INVALID_PLANT_VALUE

    sale_price = (
        expected_sale_price(
            obs,
            crop,
            harvest_days,
            expected_units
        )
    )

    expected_revenue = (
        expected_units
        * sale_price
    )

    seed_cost = (
        seed_price(
            obs,
            crop
        )
    )

    # --------------------------------------------------------
    # INITIAL WORKER TRAVEL
    # --------------------------------------------------------

    worker_positions = [
        get_worker_pos(
            farm,
            i
        )
        for i in range(
            min(
                MAX_WORKERS,
                1
                + len(
                    farm.get(
                        "hands",
                        []
                    )
                )
            )
        )
    ]

    nearest_worker_distance = min(
        manhattan(
            position,
            (x, y)
        )
        for position
        in worker_positions
    )

    # --------------------------------------------------------
    # MAINTENANCE
    # --------------------------------------------------------

    # Planting day through final harvest day.
    watering_actions = (
        harvest_days + 1
    )

    # Compact fields are cheaper to revisit.
    shed_distance = min(
        manhattan(
            (x, y),
            shed
        )
        for shed
        in SHED_ACCESS
    )

    maintenance_travel_actions = (
        watering_actions
        * min(
            shed_distance,
            5
        )
        * MAINTENANCE_TRAVEL_FACTOR
    )

    # --------------------------------------------------------
    # TOTAL LABOR OPPORTUNITY COST
    # --------------------------------------------------------

    planting_actions = 1

    harvest_actions = 1

    return_to_shed_actions = (
        shed_distance
    )

    place_actions = 1

    total_actions = (
        nearest_worker_distance
        + planting_actions
        + watering_actions
        + maintenance_travel_actions
        + harvest_actions
        + return_to_shed_actions
        + place_actions
    )

    labor_cost = (
        ACTION_OPPORTUNITY_COST
        * total_actions
    )

    net_value = (
        expected_revenue
        - seed_cost
        - labor_cost
    )

    return float(
        net_value
    )


def build_long_term_value_maps(
    obs
):
    """
    Returns:
        [5, 10, 10]

    One map for each crop.
    """

    values = np.full(
        (
            len(CROPS),
            BOARD_SIZE,
            BOARD_SIZE
        ),
        INVALID_PLANT_VALUE,
        dtype=np.float32
    )

    for crop_i, crop in enumerate(
        CROPS
    ):

        for y in range(
            BOARD_SIZE
        ):

            for x in range(
                BOARD_SIZE
            ):

                values[
                    crop_i,
                    y,
                    x
                ] = (
                    long_term_plant_value(
                        obs,
                        crop,
                        x,
                        y
                    )
                )

    return values


def crop_value_features(
    value_maps
):
    """
    Two features per crop:
      best tile value
      average profitable tile value
    """

    features = []

    for crop_i in range(
        len(CROPS)
    ):

        values = (
            value_maps[
                crop_i
            ].reshape(-1)
        )

        valid = values[
            values
            > INVALID_PLANT_VALUE / 2
        ]

        if len(valid) == 0:

            best = -500.0
            average = -500.0

        else:

            best = float(
                valid.max()
            )

            profitable = valid[
                valid > 0
            ]

            if len(profitable):

                average = float(
                    profitable.mean()
                )

            else:

                average = float(
                    valid.mean()
                )

        # Compress into roughly neural-network-friendly range.
        features += [
            np.tanh(
                best / 500.0
            ),

            np.tanh(
                average / 500.0
            ),
        ]

    return np.asarray(
        features,
        dtype=np.float32
    )


# ============================================================
# LIQUID / STRATEGIC FARM VALUE
# ============================================================

def liquid_wealth(
    obs
):

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    private = obs.get(
        "private", {}
    )

    prices = (
        obs.get(
            "market", {}
        ).get(
            "prices", {}
        )
    )

    value = float(
        farm["money"]
    )

    # Seeds retain purchase value.
    seeds = private.get(
        "seeds", {}
    )

    for crop in CROPS:

        value += (
            seeds.get(
                crop,
                0
            )
            * seed_price(
                obs,
                crop
            )
        )

    # Shed products.
    shed = private.get(
        "shed", {}
    )

    for crop in CROPS:

        value += (
            shed.get(
                crop,
                0
            )
            * prices.get(
                crop,
                0
            )
        )

    # Worker inventory.
    for inv in private.get(
        "inventories",
        []
    ):

        for crop in CROPS:

            value += (
                inv.get(
                    crop,
                    0
                )
                * prices.get(
                    crop,
                    0
                )
            )

    # Purchased land.
    for region in farm.get(
        "unlocked_quadrants",
        []
    ):

        value += (
            LAND_VALUE.get(
                region,
                0
            )
        )

    return float(
        value
    )


def existing_plant_value(
    obs,
    tile,
    x,
    y
):
    """
    Approximate remaining economic value of an existing plant.
    """

    crop = tile.get(
        "crop"
    )

    if crop not in CROPS:
        return 0.0

    age = (
        obs["day"]
        - tile.get(
            "planted_day",
            obs["day"]
        )
    )

    spec = (
        CROP_SPECS[crop]
    )

    days_remaining = (
        DAYS_PER_GAME
        - 1
        - obs["day"]
    )

    future_units, future_days = (
        expected_crop_yield(
            crop,
            min(
                days_remaining,
                max(
                    0,
                    spec[
                        "max_yield_day"
                    ]
                    + 4
                    - age
                )
            )
        )
    )

    units = max(
        float(
            tile.get(
                "yield_units",
                0
            )
        ),
        future_units
    )

    price = (
        expected_sale_price(
            obs,
            crop,
            future_days,
            units
        )
    )

    shed_distance = min(
        manhattan(
            (x, y),
            shed
        )
        for shed
        in SHED_ACCESS
    )

    future_action_cost = (
        ACTION_OPPORTUNITY_COST
        * (
            max(
                1,
                future_days
            )
            + shed_distance
            + 2
        )
    )

    return max(
        0.0,
        units * price
        - future_action_cost
    )


def strategic_value(
    obs
):
    """
    Planner reward metric.

    Unlike plain cash, this preserves value invested in:
      seeds
      inventory
      land
      growing crops
    """

    value = (
        liquid_wealth(obs)
    )

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    for y, row in enumerate(
        farm["tiles"]
    ):

        for x, tile in enumerate(
            row
        ):

            if not isinstance(
                tile,
                dict
            ):
                continue

            if tile.get(
                "kind"
            ) != "PLANT":
                continue

            value += (
                existing_plant_value(
                    obs,
                    tile,
                    x,
                    y
                )
            )

    return float(
        value
    )


# ============================================================
# SPATIAL CNN ENCODING
# ============================================================

SPATIAL_CHANNELS = 17
SPATIAL_EMBED_DIM = 128


def encode_spatial_board(
    obs
):
    """
    Tensor shape:
        [17, 10, 10]

    Channels:
      0 empty
      1 locked
      2 weed
      3-7 crop identity
      8 watered
      9 harvestable
     10 yield amount
     11 crop age
     12 farmer
     13 hands
     14 shed proximity
     15 unlocked
     16 nearest-worker distance
    """

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    board = np.zeros(
        (
            SPATIAL_CHANNELS,
            BOARD_SIZE,
            BOARD_SIZE
        ),
        dtype=np.float32
    )

    worker_positions = [
        get_worker_pos(
            farm,
            i
        )
        for i in range(
            min(
                MAX_WORKERS,
                1
                + len(
                    farm.get(
                        "hands",
                        []
                    )
                )
            )
        )
    ]

    for y in range(
        BOARD_SIZE
    ):

        for x in range(
            BOARD_SIZE
        ):

            tile = farm[
                "tiles"
            ][y][x]

            # Empty
            if tile is None:
                board[0, y, x] = 1

            # Locked
            elif tile == "LOCKED":
                board[1, y, x] = 1

            elif isinstance(
                tile,
                dict
            ):

                kind = tile.get(
                    "kind"
                )

                if kind == "WEED":

                    board[
                        2,
                        y,
                        x
                    ] = 1

                elif kind == "PLANT":

                    crop = tile.get(
                        "crop"
                    )

                    if crop in CROPS:

                        board[
                            3
                            + CROPS.index(
                                crop
                            ),
                            y,
                            x
                        ] = 1

                    board[
                        8,
                        y,
                        x
                    ] = float(
                        tile.get(
                            "watered_today",
                            False
                        )
                    )

                    board[
                        9,
                        y,
                        x
                    ] = float(
                        tile.get(
                            "yield_units",
                            0
                        ) > 0
                    )

                    board[
                        10,
                        y,
                        x
                    ] = (
                        tile.get(
                            "yield_units",
                            0
                        ) / 6.0
                    )

                    age = (
                        obs["day"]
                        - tile.get(
                            "planted_day",
                            obs["day"]
                        )
                    )

                    board[
                        11,
                        y,
                        x
                    ] = min(
                        1.0,
                        age / 20.0
                    )

            # Farmer
            fx, fy = tuple(
                farm["farmer"]
            )

            if (
                x == fx
                and y == fy
            ):
                board[
                    12,
                    y,
                    x
                ] = 1

            # Hands
            for hand in farm.get(
                "hands",
                []
            ):

                hx, hy = tuple(
                    hand
                )

                if (
                    x == hx
                    and y == hy
                ):
                    board[
                        13,
                        y,
                        x
                    ] += 1

            # Shed proximity
            shed_dist = min(
                manhattan(
                    (x, y),
                    shed
                )
                for shed
                in SHED_ACCESS
            )

            board[
                14,
                y,
                x
            ] = (
                1.0
                - min(
                    shed_dist,
                    10
                ) / 10.0
            )

            # Unlocked
            board[
                15,
                y,
                x
            ] = float(
                tile != "LOCKED"
            )

            # Distance to nearest current worker
            nearest_worker = min(
                manhattan(
                    (x, y),
                    worker
                )
                for worker
                in worker_positions
            )

            board[
                16,
                y,
                x
            ] = (
                nearest_worker
                / 18.0
            )

    return board


# ============================================================
# SPATIAL CNN
# ============================================================

class SpatialFarmCNN(
    nn.Module
):

    def __init__(
        self,
        in_channels=SPATIAL_CHANNELS,
        embedding_dim=SPATIAL_EMBED_DIM,
        num_crops=len(CROPS)
    ):

        super().__init__()

        self.encoder = (
            nn.Sequential(

                nn.Conv2d(
                    in_channels,
                    32,
                    kernel_size=3,
                    padding=1
                ),

                nn.ReLU(),

                nn.Conv2d(
                    32,
                    64,
                    kernel_size=3,
                    padding=1
                ),

                nn.ReLU(),

                nn.Conv2d(
                    64,
                    64,
                    kernel_size=3,
                    padding=1
                ),

                nn.ReLU(),
            )
        )

        # Flatten rather than average pooling so exact geometry
        # survives in the planner embedding.
        self.embedding_head = (
            nn.Sequential(

                nn.Flatten(),

                nn.Linear(
                    64
                    * BOARD_SIZE
                    * BOARD_SIZE,
                    embedding_dim
                ),

                nn.ReLU(),
            )
        )

        # One 10x10 map for each crop.
        self.plant_head = (
            nn.Conv2d(
                64,
                num_crops,
                kernel_size=1
            )
        )

    def forward(
        self,
        board
    ):

        features = (
            self.encoder(
                board
            )
        )

        embedding = (
            self.embedding_head(
                features
            )
        )

        plant_logits = (
            self.plant_head(
                features
            )
        )

        return (
            embedding,
            plant_logits
        )


# ============================================================
# CNN AUXILIARY TRAINING
#
# Long-term economic maps act as a teacher.
# CNN learns a smooth spatial policy from them.
# ============================================================

def train_spatial_model(
    spatial_net,
    optimizer,
    obs,
    value_maps,
    temperature=30.0
):

    board = torch.tensor(
        encode_spatial_board(
            obs
        ),
        dtype=torch.float32,
        device=device
    )[None]

    _, logits = (
        spatial_net(
            board
        )
    )

    value_tensor = (
        torch.tensor(
            value_maps,
            dtype=torch.float32,
            device=device
        )
    )

    losses = []

    for crop_i in range(
        len(CROPS)
    ):

        values = (
            value_tensor[
                crop_i
            ].reshape(-1)
        )

        pred = (
            logits[
                0,
                crop_i
            ].reshape(-1)
        )

        valid = (
            values
            > INVALID_PLANT_VALUE / 2
        )

        if valid.sum() < 2:
            continue

        target_distribution = (
            torch.softmax(
                values[valid]
                / temperature,
                dim=0
            )
        )

        predicted_log_probs = (
            torch.log_softmax(
                pred[valid],
                dim=0
            )
        )

        loss = -(
            target_distribution
            * predicted_log_probs
        ).sum()

        losses.append(
            loss
        )

    if not losses:
        return None

    loss = torch.stack(
        losses
    ).mean()

    optimizer.zero_grad()

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        spatial_net.parameters(),
        10
    )

    optimizer.step()

    return float(
        loss.item()
    )


def spatial_forward(
    spatial_net,
    obs
):

    board = torch.tensor(
        encode_spatial_board(
            obs
        ),
        dtype=torch.float32,
        device=device
    )[None]

    with torch.no_grad():

        embedding, logits = (
            spatial_net(
                board
            )
        )

    return (
        embedding[
            0
        ].cpu().numpy(),

        logits[
            0
        ].cpu().numpy()
    )


# ============================================================
# PLANNER STATE
# ============================================================

def board_summary(
    farm
):

    empty = 0
    locked = 0
    weeds = 0

    crop_counts = {
        crop: 0
        for crop in CROPS
    }

    harvestable = 0
    unwatered = 0

    for row in farm[
        "tiles"
    ]:

        for tile in row:

            if tile is None:

                empty += 1
                continue

            if tile == "LOCKED":

                locked += 1
                continue

            if not isinstance(
                tile,
                dict
            ):
                continue

            if tile.get(
                "kind"
            ) == "WEED":

                weeds += 1

            elif tile.get(
                "kind"
            ) == "PLANT":

                crop = tile.get(
                    "crop"
                )

                if crop in CROPS:

                    crop_counts[
                        crop
                    ] += 1

                if tile.get(
                    "yield_units",
                    0
                ) > 0:

                    harvestable += 1

                if not tile.get(
                    "watered_today",
                    False
                ):

                    unwatered += 1

    result = [

        empty / 100.0,

        locked / 100.0,

        weeds / 100.0,

        harvestable / 100.0,

        unwatered / 100.0,
    ]

    result += [
        crop_counts[crop]
        / 100.0
        for crop in CROPS
    ]

    return np.asarray(
        result,
        dtype=np.float32
    )


def encode_planner_state(
    obs,
    spatial_embedding,
    value_maps
):

    p = int(
        obs["player"]
    )

    me = obs[
        "farms"
    ][p]

    opp = obs[
        "farms"
    ][1 - p]

    private = obs.get(
        "private", {}
    )

    prices = (
        obs.get(
            "market", {}
        ).get(
            "prices", {}
        )
    )

    unlocked = me.get(
        "unlocked_quadrants",
        []
    )

    seeds = private.get(
        "seeds", {}
    )

    shed = private.get(
        "shed", {}
    )

    x = [

        obs["day"]
        / DAYS_PER_GAME,

        obs["hour"]
        / TURNS_PER_DAY,

        me["money"]
        / 10000.0,

        opp["money"]
        / 10000.0,

        len(
            me.get(
                "hands", []
            )
        ) / MAX_HANDS,
    ]

    x += [
        float(
            region
            in unlocked
        )
        for region
        in REGIONS
    ]

    x += [
        seeds.get(
            crop,
            0
        ) / 20.0
        for crop in CROPS
    ]

    x += [
        shed.get(
            crop,
            0
        ) / 100.0
        for crop in CROPS
    ]

    x += [
        prices.get(
            crop,
            0
        ) / 300.0
        for crop in CROPS
    ]

    x += list(
        board_summary(
            me
        )
    )

    x += list(
        board_summary(
            opp
        )
    )

    # Explicit long-term crop economics.
    x += list(
        crop_value_features(
            value_maps
        )
    )

    # CNN spatial representation.
    x += list(
        spatial_embedding
    )

    return np.asarray(
        x,
        dtype=np.float32
    )


# ============================================================
# PHASE-SPECIFIC GRU PLANNER
# ============================================================

class FarmPlanner(
    nn.Module
):

    def __init__(
        self,
        state_dim,
        hidden_dim=256
    ):

        super().__init__()

        self.hidden_dim = (
            hidden_dim
        )

        self.encoder = (
            nn.Sequential(

                nn.Linear(
                    state_dim,
                    256
                ),

                nn.ReLU(),

                nn.Linear(
                    256,
                    256
                ),

                nn.ReLU(),
            )
        )

        self.gru = (
            nn.GRUCell(
                256,
                hidden_dim
            )
        )

        self.hire_head = (
            nn.Linear(
                hidden_dim,
                MAX_HANDS + 1
            )
        )

        self.land_head = (
            nn.Linear(
                hidden_dim,
                2
            )
        )

        self.region_head = (
            nn.Linear(
                hidden_dim,
                MAX_WORKERS
                * len(REGIONS)
            )
        )

        self.role_head = (
            nn.Linear(
                hidden_dim,
                MAX_WORKERS
                * len(ROLES)
            )
        )

        self.crop_head = (
            nn.Linear(
                hidden_dim,
                MAX_WORKERS
                * len(CROPS)
            )
        )

        self.value_head = (
            nn.Linear(
                hidden_dim,
                1
            )
        )

    def initial_hidden(
        self,
        batch=1
    ):

        return torch.zeros(
            batch,
            self.hidden_dim,
            device=device
        )

    def forward(
        self,
        state,
        hidden
    ):

        z = self.encoder(
            state
        )

        hidden = self.gru(
            z,
            hidden
        )

        B = state.shape[0]

        return {

            "hidden":
                hidden,

            "hire":
                self.hire_head(
                    hidden
                ),

            "land":
                self.land_head(
                    hidden
                ),

            "regions":
                self.region_head(
                    hidden
                ).view(
                    B,
                    MAX_WORKERS,
                    len(REGIONS)
                ),

            "roles":
                self.role_head(
                    hidden
                ).view(
                    B,
                    MAX_WORKERS,
                    len(ROLES)
                ),

            "crops":
                self.crop_head(
                    hidden
                ).view(
                    B,
                    MAX_WORKERS,
                    len(CROPS)
                ),

            "value":
                self.value_head(
                    hidden
                ),
        }


class PlannerExperts(
    nn.Module
):

    def __init__(
        self,
        planner_state_dim,
        hidden=256
    ):

        super().__init__()

        self.experts = (
            nn.ModuleList([
                FarmPlanner(
                    planner_state_dim,
                    hidden_dim=hidden
                )
                for _ in range(
                    NUM_PHASES
                )
            ])
        )

    def get(
        self,
        phase
    ):

        return self.experts[
            phase
        ]


# ============================================================
# GRAND PLANNER
# ============================================================

def encode_grand_state(
    obs,
    value_maps
):

    p = int(
        obs["player"]
    )

    me = obs[
        "farms"
    ][p]

    opp = obs[
        "farms"
    ][1 - p]

    unlocked_shops = (
        obs.get(
            "town", {}
        ).get(
            "unlocked_shops",
            []
        )
    )

    prices = (
        obs.get(
            "market", {}
        ).get(
            "prices", {}
        )
    )

    shop_features = [
        unlocked_shops.count(
            shop
        ) / 8.0
        for shop
        in SHOPS
    ]

    x = [

        obs["day"]
        / DAYS_PER_GAME,

        len(
            unlocked_shops
        ) / 8.0,

        me["money"]
        / 10000.0,

        opp["money"]
        / 10000.0,

        len(
            me.get(
                "unlocked_quadrants",
                []
            )
        ) / 4.0,

        len(
            me.get(
                "hands",
                []
            )
        ) / MAX_HANDS,
    ]

    x += (
        shop_features
    )

    x += [
        prices.get(
            crop,
            0
        ) / 300.0
        for crop
        in CROPS
    ]

    x += list(
        crop_value_features(
            value_maps
        )
    )

    return np.asarray(
        x,
        dtype=np.float32
    )


class GrandPlanner(
    nn.Module
):

    def __init__(
        self,
        state_dim,
        hidden=128,
        num_phases=NUM_PHASES
    ):

        super().__init__()

        self.net = (
            nn.Sequential(

                nn.Linear(
                    state_dim,
                    hidden
                ),

                nn.ReLU(),

                nn.Linear(
                    hidden,
                    hidden
                ),

                nn.ReLU(),
            )
        )

        self.phase_head = (
            nn.Linear(
                hidden,
                num_phases
            )
        )

        self.value_head = (
            nn.Linear(
                hidden,
                1
            )
        )

    def forward(
        self,
        state
    ):

        z = self.net(
            state
        )

        return {

            "phase_logits":
                self.phase_head(
                    z
                ),

            "value":
                self.value_head(
                    z
                ),
        }


def choose_phase(
    grand_planner,
    obs,
    value_maps,
    deterministic=False
):

    state = torch.tensor(
        encode_grand_state(
            obs,
            value_maps
        ),
        dtype=torch.float32,
        device=device
    )[None]

    out = grand_planner(
        state
    )

    dist = Categorical(
        logits=out[
            "phase_logits"
        ]
    )

    if deterministic:

        phase = out[
            "phase_logits"
        ].argmax(
            dim=-1
        )

    else:

        phase = (
            dist.sample()
        )

    return (
        int(
            phase.item()
        ),

        dist.log_prob(
            phase
        ),

        out["value"]
    )


# ============================================================
# SPATIAL TARGET SELECTION
# ============================================================

def _standardize(
    values
):

    values = np.asarray(
        values,
        dtype=np.float32
    )

    return (
        values
        - values.mean()
    ) / (
        values.std()
        + 1e-6
    )


def best_plant_target(
    obs,
    region_idx,
    crop_idx,
    plant_logits,
    value_maps,
    cnn_weight=.50
):

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    x0, x1, y0, y1 = (
        REGION_BOUNDS[
            region_idx
        ]
    )

    candidates = []

    for y in range(
        y0,
        y1
    ):

        for x in range(
            x0,
            x1
        ):

            if farm[
                "tiles"
            ][y][x] is not None:
                continue

            value = (
                value_maps[
                    crop_idx,
                    y,
                    x
                ]
            )

            if (
                value
                <= INVALID_PLANT_VALUE / 2
            ):
                continue

            candidates.append(
                (
                    x,
                    y,
                    value,
                    plant_logits[
                        crop_idx,
                        y,
                        x
                    ]
                )
            )

    if not candidates:

        return None

    heuristic_scores = (
        _standardize([
            item[2]
            for item
            in candidates
        ])
    )

    cnn_scores = (
        _standardize([
            item[3]
            for item
            in candidates
        ])
    )

    combined = (
        (1.0 - cnn_weight)
        * heuristic_scores
        + cnn_weight
        * cnn_scores
    )

    best_i = int(
        np.argmax(
            combined
        )
    )

    x, y, value, _ = (
        candidates[
            best_i
        ]
    )

    return (
        x,
        y,
        float(value)
    )


def role_target(
    obs,
    worker_idx,
    region_idx,
    role_idx,
    crop_idx,
    plant_logits,
    value_maps,
    cnn_weight
):

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    wx, wy = (
        get_worker_pos(
            farm,
            worker_idx
        )
    )

    role = (
        ROLES[
            role_idx
        ]
    )

    inventory = {}

    inventories = (
        obs.get(
            "private", {}
        ).get(
            "inventories",
            []
        )
    )

    if worker_idx < len(
        inventories
    ):

        inventory = (
            inventories[
                worker_idx
            ]
        )

    carrying = (
        sum(
            inventory.values()
        ) > 0
    )

    # --------------------------------------------------------
    # LOGISTICS OVERRIDES EVERYTHING WHEN CARRYING
    # --------------------------------------------------------

    if carrying:

        tx, ty = min(
            SHED_ACCESS,
            key=lambda pos:
                manhattan(
                    (wx, wy),
                    pos
                )
        )

        return (
            tx,
            ty,
            0.0
        )

    x0, x1, y0, y1 = (
        REGION_BOUNDS[
            region_idx
        ]
    )

    # --------------------------------------------------------
    # HARVESTABLE
    # --------------------------------------------------------

    harvestable = []

    for y in range(
        y0,
        y1
    ):

        for x in range(
            x0,
            x1
        ):

            tile = farm[
                "tiles"
            ][y][x]

            if not isinstance(
                tile,
                dict
            ):
                continue

            if tile.get(
                "kind"
            ) != "PLANT":
                continue

            crop = tile.get(
                "crop"
            )

            if crop not in CROPS:
                continue

            age = (
                obs["day"]
                - tile.get(
                    "planted_day",
                    obs["day"]
                )
            )

            if (
                tile.get(
                    "yield_units",
                    0
                ) > 0
                and age
                >= CROP_SPECS[
                    crop
                ][
                    "first_yield_day"
                ]
            ):

                value = (
                    tile.get(
                        "yield_units",
                        0
                    )
                    * obs.get(
                        "market", {}
                    ).get(
                        "prices", {}
                    ).get(
                        crop,
                        0
                    )
                )

                harvestable.append(
                    (
                        x,
                        y,
                        value
                    )
                )

    if (
        role
        in (
            "HARVEST",
            "MAINTAIN"
        )
        and harvestable
    ):

        return max(
            harvestable,
            key=lambda item:
                item[2]
                - 2
                * manhattan(
                    (wx, wy),
                    (
                        item[0],
                        item[1]
                    )
                )
        )

    # --------------------------------------------------------
    # UNWATERED PLANTS
    # --------------------------------------------------------

    unwatered = []

    for y in range(
        y0,
        y1
    ):

        for x in range(
            x0,
            x1
        ):

            tile = farm[
                "tiles"
            ][y][x]

            if not isinstance(
                tile,
                dict
            ):
                continue

            if (
                tile.get(
                    "kind"
                ) == "PLANT"
                and not tile.get(
                    "watered_today",
                    False
                )
            ):

                crop = tile.get(
                    "crop"
                )

                value = (
                    obs.get(
                        "market", {}
                    ).get(
                        "prices", {}
                    ).get(
                        crop,
                        0
                    )
                )

                unwatered.append(
                    (
                        x,
                        y,
                        value
                    )
                )

    if (
        role
        in (
            "WATER",
            "MAINTAIN"
        )
        and unwatered
    ):

        return max(
            unwatered,
            key=lambda item:
                item[2]
                - manhattan(
                    (wx, wy),
                    (
                        item[0],
                        item[1]
                    )
                )
        )

    # --------------------------------------------------------
    # WEEDS
    # --------------------------------------------------------

    weeds = []

    for y in range(
        y0,
        y1
    ):

        for x in range(
            x0,
            x1
        ):

            tile = farm[
                "tiles"
            ][y][x]

            if (
                isinstance(
                    tile,
                    dict
                )
                and tile.get(
                    "kind"
                ) == "WEED"
            ):

                weeds.append(
                    (
                        x,
                        y,
                        0.0
                    )
                )

    if (
        role == "CLEAR"
        and weeds
    ):

        return min(
            weeds,
            key=lambda item:
                manhattan(
                    (wx, wy),
                    (
                        item[0],
                        item[1]
                    )
                )
        )

    # --------------------------------------------------------
    # PLANT TARGET
    # --------------------------------------------------------

    target = (
        best_plant_target(
            obs,
            region_idx,
            crop_idx,
            plant_logits,
            value_maps,
            cnn_weight
        )
    )

    if target is not None:
        return target

    # No useful target.
    return (
        wx,
        wy,
        0.0
    )


def select_spatial_targets(
    obs,
    regions,
    roles,
    crops,
    plant_logits,
    value_maps,
    cnn_weight
):

    target_x = []
    target_y = []
    target_values = []

    for worker_idx in range(
        MAX_WORKERS
    ):

        tx, ty, value = (
            role_target(
                obs,
                worker_idx,
                regions[
                    worker_idx
                ],
                roles[
                    worker_idx
                ],
                crops[
                    worker_idx
                ],
                plant_logits,
                value_maps,
                cnn_weight
            )
        )

        target_x.append(
            int(tx)
        )

        target_y.append(
            int(ty)
        )

        target_values.append(
            float(value)
        )

    return (
        target_x,
        target_y,
        target_values
    )


# ============================================================
# HEURISTIC WORKER-WARMUP PLAN
#
# Before the GRU is trained, give the worker a sensible teacher.
# ============================================================

def heuristic_warmup_plan(
    obs,
    plant_logits,
    value_maps
):

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    regions = []
    crops = []
    roles = []

    for worker_idx in range(
        MAX_WORKERS
    ):

        wx, wy = (
            get_worker_pos(
                farm,
                worker_idx
            )
        )

        region = (
            quadrant_index(
                wx,
                wy
            )
        )

        if not region_is_unlocked(
            farm,
            region
        ):

            region = 0

        regions.append(
            region
        )

        # Pick most profitable crop inside worker's region.
        x0, x1, y0, y1 = (
            REGION_BOUNDS[
                region
            ]
        )

        crop_scores = []

        for crop_i in range(
            len(CROPS)
        ):

            values = (
                value_maps[
                    crop_i,
                    y0:y1,
                    x0:x1
                ].reshape(-1)
            )

            valid = values[
                values
                > INVALID_PLANT_VALUE / 2
            ]

            if len(valid):

                crop_scores.append(
                    float(
                        valid.max()
                    )
                )

            else:

                crop_scores.append(
                    -1e9
                )

        crop_i = int(
            np.argmax(
                crop_scores
            )
        )

        crops.append(
            crop_i
        )

        # MAINTAIN means:
        # harvest if ready,
        # water if needed,
        # otherwise plant.
        roles.append(
            ROLES.index(
                "MAINTAIN"
            )
        )

    (
        target_x,
        target_y,
        target_values
    ) = select_spatial_targets(
        obs,
        regions,
        roles,
        crops,
        plant_logits,
        value_maps,
        cnn_weight=0.0
    )

    hire_count, buy_land = (
        apply_expansion_floor(
            obs,
            hire_count=0,
            buy_land=0
        )
    )

    return FarmPlan(

        hire_count=hire_count,

        buy_land=buy_land,

        regions=regions,

        roles=roles,

        crops=crops,

        target_x=target_x,

        target_y=target_y,

        target_values=target_values
    )


# ============================================================
# PLANNER ACTION SAMPLING
# ============================================================

def sample_plan_train(
    out,
    obs,
    plant_logits,
    value_maps,
    cnn_weight
):
    """
    Sample one planner action for PPO training.

    Forced expansion actions are treated as curriculum/teacher actions rather
    than pretending PPO sampled them. On a forced hire/land decision, that
    component is excluded from the PPO probability ratio and is instead
    learned through a small supervised teacher loss in train_planner().
    """

    def sample(logits, name):
        if not torch.isfinite(logits).all():
            raise FloatingPointError(
                f"Non-finite {name} logits before sampling. "
                "The planner weights or inputs have already become unstable."
            )

        dist = Categorical(logits=logits)
        action = dist.sample()
        return action, dist

    # --------------------------------------------------------
    # CAPITAL ACTIONS
    # --------------------------------------------------------

    raw_hire, d_hire = sample(
        out["hire"],
        "hire"
    )

    raw_land, d_land = sample(
        out["land"],
        "land"
    )

    raw_hire_i = int(raw_hire[0].item())
    raw_land_i = int(raw_land[0].item())

    final_hire, final_land = apply_expansion_floor(
        obs,
        hire_count=raw_hire_i,
        buy_land=raw_land_i
    )

    # If the curriculum changed the sampled capital action, PPO must NOT
    # pretend that this was sampled from the old policy. We record a mask
    # and learn that forced action with a separate teacher loss later.
    ppo_hire_active = (final_hire == raw_hire_i)
    ppo_land_active = (final_land == raw_land_i)

    hire = torch.full_like(raw_hire, final_hire)
    land = torch.full_like(raw_land, final_land)

    if ppo_hire_active:
        lp_hire = d_hire.log_prob(hire)
    else:
        lp_hire = torch.zeros_like(
            d_hire.log_prob(raw_hire)
        )

    if ppo_land_active:
        lp_land = d_land.log_prob(land)
    else:
        lp_land = torch.zeros_like(
            d_land.log_prob(raw_land)
        )

    # --------------------------------------------------------
    # WORKER STRATEGY ACTIONS
    # --------------------------------------------------------

    p = int(obs["player"])
    farm = obs["farms"][p]

    unlocked = farm.get(
        "unlocked_quadrants",
        ["NW"]
    )

    legal_regions = torch.tensor(
        [region in unlocked for region in REGIONS],
        dtype=torch.bool,
        device=device
    )

    if not legal_regions.any():
        legal_regions[0] = True

    region_logits = out["regions"].clone()
    region_logits[:, :, ~legal_regions] = -1e9

    regions, d_regions = sample(
        region_logits,
        "region"
    )

    roles, d_roles = sample(
        out["roles"],
        "role"
    )

    crops, d_crops = sample(
        out["crops"],
        "crop"
    )

    lp_regions = d_regions.log_prob(regions)
    lp_roles = d_roles.log_prob(roles)
    lp_crops = d_crops.log_prob(crops)

    logp = (
        lp_hire
        + lp_land
        + lp_regions.sum(dim=-1)
        + lp_roles.sum(dim=-1)
        + lp_crops.sum(dim=-1)
    )

    regions_list = regions[0].tolist()
    roles_list = roles[0].tolist()
    crops_list = crops[0].tolist()

    target_x, target_y, target_values = select_spatial_targets(
        obs,
        regions_list,
        roles_list,
        crops_list,
        plant_logits,
        value_maps,
        cnn_weight
    )

    actions = {
        "hire": int(hire[0].item()),
        "land": int(land[0].item()),
        "regions": regions_list,
        "roles": roles_list,
        "crops": crops_list,
        "legal_regions": legal_regions.detach().cpu().tolist(),

        # PPO masks: False means this capital decision was imposed by the
        # curriculum and is therefore excluded from the PPO ratio.
        "ppo_hire_active": bool(ppo_hire_active),
        "ppo_land_active": bool(ppo_land_active),
    }

    plan = FarmPlan(
        hire_count=actions["hire"],
        buy_land=actions["land"],
        regions=regions_list,
        roles=roles_list,
        crops=crops_list,
        target_x=target_x,
        target_y=target_y,
        target_values=target_values
    )

    value_scalar = out["value"][0, 0]

    if not torch.isfinite(logp).all() or not torch.isfinite(value_scalar):
        raise FloatingPointError(
            "Planner produced a non-finite log-probability or value estimate."
        )

    return (
        plan,
        actions,
        float(logp.item()),
        float(value_scalar.item())
    )

# ============================================================
# WORKER REGION TILE ENCODING
# ============================================================

WORKER_TILE_TYPES = [
    "EMPTY",
    "LOCKED",
    "WEED",
    *CROPS,
]

WORKER_TILE_DIM = (
    len(
        WORKER_TILE_TYPES
    )
    + 10
)


def encode_worker_tile(
    tile,
    x,
    y,
    wx,
    wy,
    tx,
    ty,
    day
):

    f = np.zeros(
        WORKER_TILE_DIM,
        dtype=np.float32
    )

    if tile is None:

        f[
            WORKER_TILE_TYPES.index(
                "EMPTY"
            )
        ] = 1

    elif tile == "LOCKED":

        f[
            WORKER_TILE_TYPES.index(
                "LOCKED"
            )
        ] = 1

    elif isinstance(
        tile,
        dict
    ):

        kind = tile.get(
            "kind"
        )

        if kind == "WEED":

            f[
                WORKER_TILE_TYPES.index(
                    "WEED"
                )
            ] = 1

        elif kind == "PLANT":

            crop = tile.get(
                "crop"
            )

            if crop in CROPS:

                f[
                    WORKER_TILE_TYPES.index(
                        crop
                    )
                ] = 1

            f[-10] = float(
                tile.get(
                    "watered_today",
                    False
                )
            )

            f[-9] = (
                tile.get(
                    "yield_units",
                    0
                ) / 6.0
            )

            age = (
                day
                - tile.get(
                    "planted_day",
                    day
                )
            )

            f[-8] = min(
                1.0,
                age / 20.0
            )

            f[-7] = (
                tile.get(
                    "consecutive_unwatered",
                    0
                ) / 2.0
            )

    # Spatial coordinates.
    f[-6:] = [

        x / 9.0,

        y / 9.0,

        (x - wx) / 9.0,

        (y - wy) / 9.0,

        (x - tx) / 9.0,

        (y - ty) / 9.0,
    ]

    return f


def encode_worker_region(
    obs,
    plan,
    worker_idx
):

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    wx, wy = (
        get_worker_pos(
            farm,
            worker_idx
        )
    )

    tx = (
        plan.target_x[
            worker_idx
        ]
    )

    ty = (
        plan.target_y[
            worker_idx
        ]
    )

    region_idx = (
        plan.regions[
            worker_idx
        ]
    )

    x0, x1, y0, y1 = (
        REGION_BOUNDS[
            region_idx
        ]
    )

    return np.stack([
        encode_worker_tile(

            farm[
                "tiles"
            ][y][x],

            x,
            y,

            wx,
            wy,

            tx,
            ty,

            obs["day"]
        )

        for y in range(
            y0,
            y1
        )

        for x in range(
            x0,
            x1
        )
    ])


# ============================================================
# WORKER CONTEXT
# ============================================================

def build_worker_context(
    obs,
    plan,
    worker_idx,
    planner_hidden
):

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    x, y = (
        get_worker_pos(
            farm,
            worker_idx
        )
    )

    tx = (
        plan.target_x[
            worker_idx
        ]
    )

    ty = (
        plan.target_y[
            worker_idx
        ]
    )

    target_value = (
        plan.target_values[
            worker_idx
        ]
    )

    private = obs.get(
        "private", {}
    )

    inventories = (
        private.get(
            "inventories",
            []
        )
    )

    if worker_idx < len(
        inventories
    ):

        inventory = (
            inventories[
                worker_idx
            ]
        )

    else:

        inventory = {}

    prices = (
        obs.get(
            "market", {}
        ).get(
            "prices", {}
        )
    )

    context = list(
        planner_hidden
    )

    context += list(
        one_hot(
            plan.regions[
                worker_idx
            ],
            len(REGIONS)
        )
    )

    context += list(
        one_hot(
            plan.roles[
                worker_idx
            ],
            len(ROLES)
        )
    )

    context += list(
        one_hot(
            plan.crops[
                worker_idx
            ],
            len(CROPS)
        )
    )

    # 9 scalar features.
    context += [

        obs["day"]
        / DAYS_PER_GAME,

        obs["hour"]
        / TURNS_PER_DAY,

        x / 9.0,

        y / 9.0,

        tx / 9.0,

        ty / 9.0,

        (tx - x) / 9.0,

        (ty - y) / 9.0,

        np.tanh(
            target_value
            / 500.0
        ),
    ]

    context += [
        inventory.get(
            crop,
            0
        ) / 20.0
        for crop
        in CROPS
    ]

    context += [
        prices.get(
            crop,
            0
        ) / 300.0
        for crop
        in CROPS
    ]

    return np.asarray(
        context,
        dtype=np.float32
    )


def make_worker_batch(
    obs,
    plan,
    hidden_np
):

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    n_workers = min(
        MAX_WORKERS,
        1
        + len(
            farm.get(
                "hands",
                []
            )
        )
    )

    tiles = []
    contexts = []

    for worker_idx in range(
        n_workers
    ):

        tiles.append(
            encode_worker_region(
                obs,
                plan,
                worker_idx
            )
        )

        contexts.append(
            build_worker_context(
                obs,
                plan,
                worker_idx,
                hidden_np
            )
        )

    return (
        np.stack(
            tiles
        ),

        np.stack(
            contexts
        )
    )


# ============================================================
# WORKER TRANSFORMER
# ============================================================

class WorkerTransformer(
    nn.Module
):

    def __init__(
        self,
        tile_dim,
        context_dim,
        d_model=128
    ):

        super().__init__()

        self.tile_emb = (
            nn.Linear(
                tile_dim,
                d_model
            )
        )

        self.context_emb = (
            nn.Linear(
                context_dim,
                d_model
            )
        )

        self.cls = (
            nn.Parameter(
                torch.zeros(
                    1,
                    1,
                    d_model
                )
            )
        )

        self.pos = (
            nn.Parameter(
                torch.randn(
                    1,
                    26,
                    d_model
                )
                * .02
            )
        )

        layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,

                nhead=4,

                dim_feedforward=512,

                batch_first=True,

                norm_first=True
            )
        )

        self.transformer = (
            nn.TransformerEncoder(
                layer,
                num_layers=3
            )
        )

        self.q_head = (
            nn.Linear(
                d_model,
                len(
                    WORKER_ACTIONS
                )
            )
        )

    def forward(
        self,
        tiles,
        context
    ):

        B = (
            tiles.shape[0]
        )

        tile_tokens = (
            self.tile_emb(
                tiles
            )
        )

        cls = (
            self.cls.expand(
                B,
                -1,
                -1
            )
            + self.context_emb(
                context
            ).unsqueeze(1)
        )

        x = torch.cat(
            [
                cls,
                tile_tokens
            ],
            dim=1
        )

        x = (
            self.transformer(
                x + self.pos
            )
        )

        return (
            self.q_head(
                x[:, 0]
            )
        )


# ============================================================
# LEGAL WORKER ACTIONS
# ============================================================

def worker_mask(
    obs,
    worker_idx
):

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    x, y = (
        get_worker_pos(
            farm,
            worker_idx
        )
    )

    tile = farm[
        "tiles"
    ][y][x]

    mask = np.zeros(
        len(
            WORKER_ACTIONS
        ),
        dtype=bool
    )

    # PASS
    mask[0] = True

    moves = [
        (1, 0, -1),   # north
        (2, 0, 1),    # south
        (3, 1, 0),    # east
        (4, -1, 0),   # west
    ]

    for action_i, dx, dy in moves:

        nx = x + dx
        ny = y + dy

        if not (
            0 <= nx < BOARD_SIZE
            and
            0 <= ny < BOARD_SIZE
        ):
            continue

        destination = farm[
            "tiles"
        ][ny][nx]

        if destination != "LOCKED":

            mask[
                action_i
            ] = True

    # --------------------------------------------------------
    # TILE ACTIONS
    # --------------------------------------------------------

    if isinstance(
        tile,
        dict
    ):

        kind = tile.get(
            "kind"
        )

        if kind == "WEED":

            # DIG
            mask[7] = True

        elif kind == "PLANT":

            # WATER
            if not tile.get(
                "watered_today",
                False
            ):

                mask[5] = True

            crop = tile.get(
                "crop"
            )

            age = (
                obs["day"]
                - tile.get(
                    "planted_day",
                    obs["day"]
                )
            )

            # HARVEST only after crop is mature.
            if (
                crop in CROP_SPECS
                and
                age
                >= CROP_SPECS[
                    crop
                ][
                    "first_yield_day"
                ]
                and
                tile.get(
                    "yield_units",
                    0
                ) > 0
            ):

                mask[6] = True

    # --------------------------------------------------------
    # PLANT
    # --------------------------------------------------------

    if tile is None:

        seeds = (
            obs.get(
                "private", {}
            ).get(
                "seeds", {}
            )
        )

        for crop_i, crop in enumerate(
            CROPS
        ):

            mask[
                8 + crop_i
            ] = (
                seeds.get(
                    crop,
                    0
                ) > 0
            )

    # --------------------------------------------------------
    # PLACE INTO SHED
    # --------------------------------------------------------

    inventories = (
        obs.get(
            "private", {}
        ).get(
            "inventories",
            []
        )
    )

    if (
        worker_idx
        < len(
            inventories
        )
        and
        is_shed_access(
            x,
            y
        )
    ):

        carrying = (
            sum(
                inventories[
                    worker_idx
                ].values()
            ) > 0
        )

        if carrying:

            mask[13] = True

    return mask


# ============================================================
# ACTION CONVERSION
# ============================================================

def carried_item_for_worker(
    obs,
    worker_idx
):

    inventories = (
        obs.get(
            "private", {}
        ).get(
            "inventories",
            []
        )
    )

    if worker_idx >= len(
        inventories
    ):

        return None

    inventory = (
        inventories[
            worker_idx
        ]
    )

    if not inventory:

        return None

    prices = (
        obs.get(
            "market", {}
        ).get(
            "prices", {}
        )
    )

    available = [
        (
            item,
            amount
        )
        for item, amount
        in inventory.items()
        if amount > 0
    ]

    if not available:

        return None

    # Deposit highest-value inventory first.
    item, amount = max(
        available,
        key=lambda pair:
            prices.get(
                pair[0],
                0
            )
            * pair[1]
    )

    return (
        item,
        int(amount)
    )


def worker_action_to_kaggle(
    action_idx,
    obs,
    worker_idx
):

    op, arg = (
        WORKER_ACTIONS[
            action_idx
        ]
    )

    if op == "PLANT":

        return [
            "PLANT",
            arg
        ]

    if op == "PLACE":

        carried = (
            carried_item_for_worker(
                obs,
                worker_idx
            )
        )

        if carried is None:

            return [
                "PASS"
            ]

        item, amount = (
            carried
        )

        return [
            "PLACE",
            item,
            amount
        ]

    return [
        op
    ]


# ============================================================
# MARKET EXECUTION
# ============================================================

MIN_CASH_RESERVE = 500.0
MAX_SEED_SPEND_FRACTION = 0.25

# ============================================================
# EXPANSION FLOOR / TEACHER POLICY
#
# Kaggriculture hands are temporary: they disappear at the end
# of each day and must be hired again the next morning. Under
# the default competition configuration, same-day hire costs
# follow Fibonacci: 1, 1, 2, 3, ... coins.
#
# We impose a small strategic floor while still letting the
# neural planner choose MORE workers or the final land quadrant.
# ============================================================

FORCED_MIN_DAILY_HANDS = 2
FORCED_MIN_REGIONS = 3          # NW + two purchased quadrants
FORCED_LAND_LAST_DAY = 18      # avoid late capital expansion
FORCED_LAND_CASH_RESERVE = 750.0

# Advanced Kaggriculture unlock order is NE, SW, SE.
LAND_PURCHASE_PRICES = [1000.0, 2000.0, 4000.0]

# Default competition configuration.
FARM_HAND_COST_MULT = 1.0


def _fib_hire_cost(
    hires_already_today,
    multiplier=FARM_HAND_COST_MULT
):
    """Cost of the next HIRE order under Kaggriculture rules."""

    n = max(
        0,
        int(hires_already_today)
    )

    if n <= 1:
        fib = 1
    else:
        a, b = 1, 1

        for _ in range(2, n + 1):
            a, b = b, a + b

        fib = b

    return float(
        multiplier * fib
    )


def estimated_hire_batch_cost(
    starting_hires_today,
    count
):
    """Estimated total cost for `count` sequential HIRE orders."""

    return sum(
        _fib_hire_cost(
            starting_hires_today + i
        )
        for i in range(
            max(0, int(count))
        )
    )


def next_land_purchase_cost(
    farm
):
    """Return the exact next land price implied by current unlock count."""

    unlocked_count = len(
        farm.get(
            "unlocked_quadrants",
            ["NW"]
        )
    )

    extra_unlocked = max(
        0,
        unlocked_count - 1
    )

    if extra_unlocked >= len(
        LAND_PURCHASE_PRICES
    ):
        return None

    return float(
        LAND_PURCHASE_PRICES[
            extra_unlocked
        ]
    )


def apply_expansion_floor(
    obs,
    hire_count,
    buy_land
):
    """
    Apply a small domain-knowledge floor to planner capital decisions.

    - Hire at least two hands every day. Hands are temporary and cheap.
    - Force land purchases until three total regions are unlocked, but
      only when the exact next land price can be paid while preserving
      a reserve and only during the first 18 days.
    - The learned planner can still choose 3-4 hands and can still buy
      the fourth quadrant on its own.
    """

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    hire_count = int(
        np.clip(
            int(hire_count),
            0,
            MAX_HANDS
        )
    )

    buy_land = int(
        bool(buy_land)
    )

    # Hands vanish at end-of-day, so this is a DAILY minimum.
    hire_count = max(
        hire_count,
        min(
            FORCED_MIN_DAILY_HANDS,
            MAX_HANDS
        )
    )

    unlocked_count = len(
        farm.get(
            "unlocked_quadrants",
            ["NW"]
        )
    )

    if unlocked_count >= len(REGIONS):
        buy_land = 0

    elif (
        obs["day"] <= FORCED_LAND_LAST_DAY
        and
        unlocked_count < FORCED_MIN_REGIONS
    ):

        next_cost = (
            next_land_purchase_cost(
                farm
            )
        )

        money = float(
            farm.get(
                "money",
                0.0
            )
        )

        if (
            next_cost is not None
            and
            money >= (
                next_cost
                + FORCED_LAND_CASH_RESERVE
            )
        ):
            buy_land = 1

    return (
        hire_count,
        buy_land
    )


def planner_market_actions(
    obs,
    plan
):
    """
    Budget-aware market policy used by both learner and frozen agents.

    Important corrections for the 30-day Kaggriculture environment:
      - Hands are temporary and must be re-hired every day.
      - HIRE costs follow 1, 1, 2, 3, ... within each day.
      - The planner's hire_count is executed as an actual desired hand count
        instead of being collapsed to one HIRE order.
      - Land costs are 1000 / 2000 / 4000 in unlock order.
      - The expansion floor guarantees at least two daily hands and forces
        the first two land purchases when they are affordable early enough.
    """

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    money = float(
        farm.get(
            "money",
            0.0
        )
    )

    private = obs.get(
        "private",
        {}
    )

    shed = private.get(
        "shed",
        {}
    )

    seeds = private.get(
        "seeds",
        {}
    )

    orders = []

    # --------------------------------------------------------
    # SELL FIRST
    # --------------------------------------------------------

    for crop in CROPS:

        amount = int(
            shed.get(
                crop,
                0
            )
        )

        if amount > 0:

            orders.append([
                "SELL",
                crop,
                amount
            ])

    # Purchases only at the beginning of a day.
    if obs["hour"] != 0:

        return orders[:10]

    # --------------------------------------------------------
    # APPLY EXPANSION FLOOR TO THIS DAY'S PLAN
    # --------------------------------------------------------

    desired_hands, want_land = (
        apply_expansion_floor(
            obs,
            hire_count=plan.hire_count,
            buy_land=plan.buy_land
        )
    )

    # --------------------------------------------------------
    # SAFE BUDGET
    # --------------------------------------------------------

    spendable_money = max(
        0.0,
        money
        - MIN_CASH_RESERVE
    )

    remaining_budget = (
        spendable_money
    )

    # --------------------------------------------------------
    # LAND
    # --------------------------------------------------------

    land_cost = (
        next_land_purchase_cost(
            farm
        )
    )

    if (
        want_land
        and
        land_cost is not None
        and
        remaining_budget >= land_cost
    ):

        orders.append([
            "BUY_LAND"
        ])

        remaining_budget = max(
            0.0,
            remaining_budget
            - land_cost
        )

    # --------------------------------------------------------
    # HIRING
    #
    # Hands disappear at the end of every day. We therefore
    # interpret planner.hire_count as today's desired number
    # of hands and emit enough HIRE orders to reach that count.
    # --------------------------------------------------------

    current_hands = len(
        farm.get(
            "hands",
            []
        )
    )

    desired_hands = int(
        np.clip(
            desired_hands,
            0,
            MAX_HANDS
        )
    )

    hires_to_place = max(
        0,
        desired_hands
        - current_hands
    )

    hires_already_today = int(
        farm.get(
            "hires_today",
            current_hands
        )
    )

    hires_placed = 0

    for hire_i in range(
        hires_to_place
    ):

        hire_cost = (
            _fib_hire_cost(
                hires_already_today
                + hire_i
            )
        )

        if remaining_budget < hire_cost:
            break

        if len(orders) >= 10:
            break

        orders.append([
            "HIRE"
        ])

        remaining_budget -= (
            hire_cost
        )

        hires_placed += 1

    # --------------------------------------------------------
    # SEED BUDGET
    # --------------------------------------------------------

    seed_budget = min(
        remaining_budget,
        spendable_money
        * MAX_SEED_SPEND_FRACTION
    )

    remaining_seed_budget = (
        seed_budget
    )

    desired = Counter()

    actual_workers = min(
        MAX_WORKERS,
        1
        + current_hands
        + hires_placed
    )

    for i in range(
        actual_workers
    ):

        role = ROLES[
            plan.roles[i]
        ]

        if role not in (
            "PLANT",
            "MAINTAIN"
        ):

            continue

        if (
            plan.target_values[i]
            <= 0
        ):

            continue

        desired[
            plan.crops[i]
        ] += 1

    for crop_i, worker_count in (
        desired.items()
    ):

        crop = CROPS[
            crop_i
        ]

        cost_per_seed = max(
            1.0,
            seed_price(
                obs,
                crop
            )
        )

        target_seed_count = (
            worker_count
            * 2
        )

        missing = max(
            0,
            target_seed_count
            - seeds.get(
                crop,
                0
            )
        )

        affordable = int(
            remaining_seed_budget
            // cost_per_seed
        )

        amount = min(
            missing,
            affordable
        )

        if amount <= 0:

            continue

        if len(orders) >= 10:
            break

        orders.append([
            "BUY_SEED",
            crop,
            int(amount)
        ])

        remaining_seed_budget -= (
            amount
            * cost_per_seed
        )

        if (
            remaining_seed_budget
            <= 0
        ):

            break

    return orders[:10]


def make_kaggle_action(
    obs,
    plan,
    worker_actions
):

    farmer_action = (
        worker_action_to_kaggle(
            worker_actions[0],
            obs,
            0
        )
    )

    hand_actions = []

    for i, action_i in enumerate(
        worker_actions[
            1:
        ],
        start=1
    ):

        hand_actions.append(
            worker_action_to_kaggle(
                action_i,
                obs,
                i
            )
        )

    return {

        "farmer":
            farmer_action,

        "hands":
            hand_actions,

        "market":
            planner_market_actions(
                obs,
                plan
            )
    }


# ============================================================
# WORKER REWARD
# ============================================================

def worker_reward(
    obs,
    next_obs,
    plan,
    worker_idx,
    action_idx
):

    p = int(
        obs["player"]
    )

    before = obs[
        "farms"
    ][p]

    after = next_obs[
        "farms"
    ][p]

    old_x, old_y = (
        get_worker_pos(
            before,
            worker_idx
        )
    )

    new_x, new_y = (
        get_worker_pos(
            after,
            worker_idx
        )
    )

    tx = (
        plan.target_x[
            worker_idx
        ]
    )

    ty = (
        plan.target_y[
            worker_idx
        ]
    )

    crop_goal = (
        CROPS[
            plan.crops[
                worker_idx
            ]
        ]
    )

    op, arg = (
        WORKER_ACTIONS[
            action_idx
        ]
    )

    reward = -0.005

    # --------------------------------------------------------
    # MOVEMENT TOWARD PLANNER TARGET
    # --------------------------------------------------------

    if op in (
        "NORTH",
        "SOUTH",
        "EAST",
        "WEST"
    ):

        old_distance = (
            manhattan(
                (
                    old_x,
                    old_y
                ),
                (
                    tx,
                    ty
                )
            )
        )

        new_distance = (
            manhattan(
                (
                    new_x,
                    new_y
                ),
                (
                    tx,
                    ty
                )
            )
        )

        reward += (
            0.05
            * (
                old_distance
                - new_distance
            )
        )

    # --------------------------------------------------------
    # PLANT
    # --------------------------------------------------------

    elif op == "PLANT":

        new_tile = (
            after[
                "tiles"
            ][old_y][old_x]
        )

        successful = (
            isinstance(
                new_tile,
                dict
            )
            and
            new_tile.get(
                "kind"
            ) == "PLANT"
            and
            new_tile.get(
                "crop"
            ) == arg
        )

        if successful:

            reward += 0.30

            if arg == crop_goal:

                reward += 0.50

            if (
                old_x == tx
                and
                old_y == ty
            ):

                reward += 1.00

    # --------------------------------------------------------
    # WATER
    # --------------------------------------------------------

    elif op == "WATER":

        old_tile = (
            before[
                "tiles"
            ][old_y][old_x]
        )

        new_tile = (
            after[
                "tiles"
            ][old_y][old_x]
        )

        if (
            isinstance(
                old_tile,
                dict
            )
            and
            isinstance(
                new_tile,
                dict
            )
            and
            not old_tile.get(
                "watered_today",
                False
            )
            and
            new_tile.get(
                "watered_today",
                False
            )
        ):

            reward += 0.50

    # --------------------------------------------------------
    # HARVEST
    # --------------------------------------------------------

    elif op == "HARVEST":

        old_tile = (
            before[
                "tiles"
            ][old_y][old_x]
        )

        new_tile = (
            after[
                "tiles"
            ][old_y][old_x]
        )

        old_yield = (
            old_tile.get(
                "yield_units",
                0
            )
            if isinstance(
                old_tile,
                dict
            )
            else 0
        )

        new_yield = (
            new_tile.get(
                "yield_units",
                0
            )
            if isinstance(
                new_tile,
                dict
            )
            else 0
        )

        if (
            new_yield
            < old_yield
            or new_tile is None
        ):

            reward += 1.00

            crop = (
                old_tile.get(
                    "crop"
                )
                if isinstance(
                    old_tile,
                    dict
                )
                else None
            )

            if crop in CROPS:

                reward += (
                    0.002
                    * old_yield
                    * obs.get(
                        "market", {}
                    ).get(
                        "prices", {}
                    ).get(
                        crop,
                        0
                    )
                )

    # --------------------------------------------------------
    # DIG WEED
    # --------------------------------------------------------

    elif op == "DIG":

        old_tile = (
            before[
                "tiles"
            ][old_y][old_x]
        )

        new_tile = (
            after[
                "tiles"
            ][old_y][old_x]
        )

        if (
            isinstance(
                old_tile,
                dict
            )
            and
            old_tile.get(
                "kind"
            ) == "WEED"
            and
            new_tile is None
        ):

            reward += 0.30

    # --------------------------------------------------------
    # PLACE INTO SHED
    # --------------------------------------------------------

    elif op == "PLACE":

        before_inv = (
            obs.get(
                "private", {}
            ).get(
                "inventories",
                []
            )
        )

        after_inv = (
            next_obs.get(
                "private", {}
            ).get(
                "inventories",
                []
            )
        )

        before_amount = 0
        after_amount = 0

        if worker_idx < len(
            before_inv
        ):

            before_amount = sum(
                before_inv[
                    worker_idx
                ].values()
            )

        if worker_idx < len(
            after_inv
        ):

            after_amount = sum(
                after_inv[
                    worker_idx
                ].values()
            )

        before_shed = sum(
            obs.get(
                "private", {}
            ).get(
                "shed", {}
            ).values()
        )

        after_shed = sum(
            next_obs.get(
                "private", {}
            ).get(
                "shed", {}
            ).values()
        )

        if (
            after_amount
            < before_amount
            and
            after_shed
            > before_shed
        ):

            reward += 0.80

    elif op == "PASS":

        reward -= 0.01

    # --------------------------------------------------------
    # SMALL TEAM-ECONOMIC SIGNAL
    # --------------------------------------------------------

    economic_delta = (
        liquid_wealth(
            next_obs
        )
        - liquid_wealth(
            obs
        )
    )

    reward += (
        0.02
        * np.tanh(
            economic_delta
            / 100.0
        )
    )

    return float(
        reward
    )


# ============================================================
# REPLAY BUFFER
# ============================================================

class ReplayBuffer:

    def __init__(
        self,
        size=150_000
    ):

        self.data = (
            deque(
                maxlen=size
            )
        )

    def add(
        self,
        *transition
    ):

        self.data.append(
            transition
        )

    def sample(
        self,
        n
    ):

        return random.sample(
            self.data,
            n
        )

    def __len__(
        self
    ):

        return len(
            self.data
        )


# ============================================================
# WORKER ACTION SELECTION
# ============================================================

def select_worker_actions(
    net,
    obs,
    tiles,
    contexts,
    epsilon
):

    tile_tensor = torch.tensor(
        tiles,
        dtype=torch.float32,
        device=device
    )

    context_tensor = torch.tensor(
        contexts,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():

        q = net(
            tile_tensor,
            context_tensor
        )

    actions = []

    for worker_idx in range(
        len(tiles)
    ):

        mask = worker_mask(
            obs,
            worker_idx
        )

        legal = np.where(
            mask
        )[0]

        if (
            random.random()
            < epsilon
        ):

            action = int(
                np.random.choice(
                    legal
                )
            )

        else:

            values = (
                q[
                    worker_idx
                ].clone()
            )

            invalid = (
                torch.tensor(
                    ~mask,
                    dtype=torch.bool,
                    device=device
                )
            )

            values[
                invalid
            ] = -1e9

            action = int(
                values.argmax().item()
            )

        actions.append(
            action
        )

    return actions


# ============================================================
# DQN WORKER UPDATE
# ============================================================

def train_worker(
    policy,
    target,
    optimizer,
    replay,
    batch_size=128,
    gamma=.99
):

    if len(
        replay
    ) < batch_size:

        return None

    batch = (
        replay.sample(
            batch_size
        )
    )

    (
        sT,
        sC,
        A,
        R,
        nT,
        nC,
        D,
        NM
    ) = zip(
        *batch
    )

    sT = torch.tensor(
        np.stack(
            sT
        ),
        dtype=torch.float32,
        device=device
    )

    sC = torch.tensor(
        np.stack(
            sC
        ),
        dtype=torch.float32,
        device=device
    )

    nT = torch.tensor(
        np.stack(
            nT
        ),
        dtype=torch.float32,
        device=device
    )

    nC = torch.tensor(
        np.stack(
            nC
        ),
        dtype=torch.float32,
        device=device
    )

    A = torch.tensor(
        A,
        dtype=torch.long,
        device=device
    )

    R = torch.tensor(
        R,
        dtype=torch.float32,
        device=device
    )

    D = torch.tensor(
        D,
        dtype=torch.float32,
        device=device
    )

    NM = torch.tensor(
        np.stack(
            NM
        ),
        dtype=torch.bool,
        device=device
    )

    q = (
        policy(
            sT,
            sC
        )
        .gather(
            1,
            A[:, None]
        )
        .squeeze(1)
    )

    with torch.no_grad():

        next_q = (
            target(
                nT,
                nC
            )
        )

        next_q[
            ~NM
        ] = -1e9

        target_q = (
            R
            + gamma
            * (
                1.0 - D
            )
            * next_q.max(
                dim=1
            ).values
        )

    loss = (
        F.smooth_l1_loss(
            q,
            target_q
        )
    )

    optimizer.zero_grad()

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        policy.parameters(),
        10
    )

    optimizer.step()

    return float(
        loss.item()
    )


# ============================================================
# PHASE-PLANNER PPO
# ============================================================

def train_planner(
    planner,
    optimizer,
    trajectory,
    gamma=.995,
    clip=.20,
    epochs=4,
    teacher_weight=.10
):
    """
    Numerically-stable PPO update for one phase planner.

    Forced hire/land curriculum actions are excluded from the PPO importance
    ratio and learned with a small supervised cross-entropy teacher loss.
    This prevents the forced action from creating an invalid PPO ratio while
    still teaching the capital heads to internalize the desired expansion.
    """

    if len(trajectory) < 2:
        return None

    # --------------------------------------------------------
    # DISCOUNTED DAILY RETURNS
    # --------------------------------------------------------

    returns_list = []
    R = 0.0

    for record in reversed(trajectory):
        reward = float(record["reward"])

        if not np.isfinite(reward):
            reward = 0.0

        R = reward + gamma * R
        returns_list.append(R)

    returns_list.reverse()

    states_np = np.stack([
        record["state"]
        for record in trajectory
    ]).astype(np.float32)

    hidden_np = np.stack([
        record["hidden"]
        for record in trajectory
    ]).astype(np.float32)

    # Defensive input sanitation. This handles an occasional environment
    # feature anomaly without allowing NaNs/Infs to enter the network.
    states_np = np.nan_to_num(
        states_np,
        nan=0.0,
        posinf=1e6,
        neginf=-1e6
    )

    hidden_np = np.nan_to_num(
        hidden_np,
        nan=0.0,
        posinf=1e6,
        neginf=-1e6
    )

    states = torch.tensor(
        states_np,
        dtype=torch.float32,
        device=device
    )

    hidden = torch.tensor(
        hidden_np,
        dtype=torch.float32,
        device=device
    )

    old_logp = torch.tensor(
        [record["logp"] for record in trajectory],
        dtype=torch.float32,
        device=device
    )

    old_values = torch.tensor(
        [record["value"] for record in trajectory],
        dtype=torch.float32,
        device=device
    )

    returns = torch.tensor(
        returns_list,
        dtype=torch.float32,
        device=device
    )

    old_logp = torch.nan_to_num(
        old_logp,
        nan=0.0,
        posinf=20.0,
        neginf=-20.0
    )

    old_values = torch.nan_to_num(
        old_values,
        nan=0.0,
        posinf=1e6,
        neginf=-1e6
    )

    returns = torch.nan_to_num(
        returns,
        nan=0.0,
        posinf=1e6,
        neginf=-1e6
    )

    # A very large value target can dominate the entire PPO update. The
    # competitive terminal bonus is small, so this ceiling is intentionally
    # generous while still preventing an accidental explosion.
    returns = torch.clamp(
        returns,
        -10000.0,
        10000.0
    )

    hire = torch.tensor(
        [record["action"]["hire"] for record in trajectory],
        dtype=torch.long,
        device=device
    )

    land = torch.tensor(
        [record["action"]["land"] for record in trajectory],
        dtype=torch.long,
        device=device
    )

    regions = torch.tensor(
        [record["action"]["regions"] for record in trajectory],
        dtype=torch.long,
        device=device
    )

    roles = torch.tensor(
        [record["action"]["roles"] for record in trajectory],
        dtype=torch.long,
        device=device
    )

    crops = torch.tensor(
        [record["action"]["crops"] for record in trajectory],
        dtype=torch.long,
        device=device
    )

    legal_regions = torch.tensor(
        [record["action"]["legal_regions"] for record in trajectory],
        dtype=torch.bool,
        device=device
    )

    # Backward-compatible defaults let an old in-memory trajectory finish
    # safely if it was created before these mask fields existed.
    ppo_hire_active = torch.tensor(
        [
            bool(record["action"].get("ppo_hire_active", True))
            for record in trajectory
        ],
        dtype=torch.float32,
        device=device
    )

    ppo_land_active = torch.tensor(
        [
            bool(record["action"].get("ppo_land_active", True))
            for record in trajectory
        ],
        dtype=torch.float32,
        device=device
    )

    advantage = returns - old_values

    advantage = torch.nan_to_num(
        advantage,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    advantage = (
        advantage - advantage.mean()
    ) / (
        advantage.std(unbiased=False) + 1e-8
    )

    last_loss = None

    for _ in range(epochs):
        out = planner(states, hidden)

        # Never allow a corrupted network to be stepped further.
        for name in ("hire", "land", "regions", "roles", "crops", "value"):
            if not torch.isfinite(out[name]).all():
                optimizer.zero_grad(set_to_none=True)
                print(
                    f"WARNING: skipped planner PPO update because {name} "
                    "contained NaN/Inf."
                )
                return None

        region_logits = out["regions"].masked_fill(
            ~legal_regions[:, None, :],
            -1e9
        )

        d_hire = Categorical(logits=out["hire"])
        d_land = Categorical(logits=out["land"])
        d_region = Categorical(logits=region_logits)
        d_role = Categorical(logits=out["roles"])
        d_crop = Categorical(logits=out["crops"])

        hire_logp = d_hire.log_prob(hire)
        land_logp = d_land.log_prob(land)

        new_logp = (
            hire_logp * ppo_hire_active
            + land_logp * ppo_land_active
            + d_region.log_prob(regions).sum(dim=1)
            + d_role.log_prob(roles).sum(dim=1)
            + d_crop.log_prob(crops).sum(dim=1)
        )

        # Exclude forced capital heads from PPO entropy as well. Their
        # supervised teacher loss below handles those timesteps instead.
        entropy_per_step = (
            d_hire.entropy() * ppo_hire_active
            + d_land.entropy() * ppo_land_active
            + d_region.entropy().sum(dim=1)
            + d_role.entropy().sum(dim=1)
            + d_crop.entropy().sum(dim=1)
        )

        entropy = entropy_per_step.mean()

        # IMPORTANT: clamp the log-ratio BEFORE exponentiating. This avoids
        # exp(large_number) -> inf, which was another route to NaN weights.
        log_ratio = torch.clamp(
            new_logp - old_logp,
            -20.0,
            20.0
        )

        ratio = torch.exp(log_ratio)

        clipped_ratio = torch.clamp(
            ratio,
            1.0 - clip,
            1.0 + clip
        )

        policy_loss = -torch.min(
            ratio * advantage,
            clipped_ratio * advantage
        ).mean()

        # Huber loss is substantially more robust than MSE to a large daily
        # strategic-value jump after buying land.
        value_loss = F.smooth_l1_loss(
            out["value"].squeeze(1),
            returns
        )

        # ----------------------------------------------------
        # CAPITAL TEACHER LOSS
        # ----------------------------------------------------
        # On curriculum-forced timesteps, explicitly teach the hire/land
        # heads the action we imposed. This is supervised learning, not PPO.

        teacher_terms = []

        forced_hire_mask = ppo_hire_active < 0.5
        forced_land_mask = ppo_land_active < 0.5

        if forced_hire_mask.any():
            teacher_terms.append(
                F.cross_entropy(
                    out["hire"][forced_hire_mask],
                    hire[forced_hire_mask]
                )
            )

        if forced_land_mask.any():
            teacher_terms.append(
                F.cross_entropy(
                    out["land"][forced_land_mask],
                    land[forced_land_mask]
                )
            )

        if teacher_terms:
            capital_teacher_loss = torch.stack(teacher_terms).mean()
        else:
            capital_teacher_loss = torch.zeros((), device=device)

        loss = (
            policy_loss
            + 0.5 * value_loss
            - 0.01 * entropy
            + teacher_weight * capital_teacher_loss
        )

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            print(
                "WARNING: skipped non-finite planner PPO loss before "
                "optimizer.step()."
            )
            return None

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            planner.parameters(),
            5.0
        )

        if not torch.isfinite(torch.as_tensor(grad_norm)):
            optimizer.zero_grad(set_to_none=True)
            print(
                "WARNING: skipped planner optimizer step because gradient "
                "norm was NaN/Inf."
            )
            return None

        optimizer.step()
        last_loss = float(loss.item())

    return last_loss

# ============================================================
# MODEL SNAPSHOTS / CHECKPOINTS
# ============================================================

def cpu_state_dict(model):
    """
    Copy model weights onto CPU so snapshots do not hold
    references to the actively training network.
    """
    return {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }


def make_policy_snapshot(
    spatial_net,
    planners,
    grand_planner,
    worker,
    game=None,
    global_step=None
):
    """
    Inference-only snapshot used for:
      - champion opponents
      - challenger evaluation
      - Kaggle submission export
    """

    return {
        "spatial_net":
            cpu_state_dict(spatial_net),

        "planners":
            cpu_state_dict(planners),

        "grand_planner":
            cpu_state_dict(grand_planner),

        "worker":
            cpu_state_dict(worker),

        "game":
            game,

        "global_step":
            global_step,
    }


def save_policy_snapshot(
    path,
    snapshot
):
    os.makedirs(
        os.path.dirname(path),
        exist_ok=True
    )

    torch.save(
        snapshot,
        path
    )


def load_policy_snapshot(
    path
):
    return torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )


# ============================================================
# COMPLETE TRAINING CHECKPOINT
# ============================================================

def save_training_checkpoint(
    path,
    game,
    global_step,
    spatial_net,
    planners,
    grand_planner,
    worker,
    target_worker,
    spatial_opt,
    planner_opt,
    grand_opt,
    worker_opt,
    score_history,
    champion_game=None
):

    os.makedirs(
        os.path.dirname(path),
        exist_ok=True
    )

    checkpoint = {

        "game":
            game,

        "global_step":
            global_step,

        "spatial_net":
            spatial_net.state_dict(),

        "planners":
            planners.state_dict(),

        "grand_planner":
            grand_planner.state_dict(),

        "worker":
            worker.state_dict(),

        "target_worker":
            target_worker.state_dict(),

        "spatial_opt":
            spatial_opt.state_dict(),

        "planner_opt":
            planner_opt.state_dict(),

        "grand_opt":
            grand_opt.state_dict(),

        "worker_opt":
            worker_opt.state_dict(),

        "score_history":
            list(score_history),

        "champion_game":
            champion_game,
    }

    torch.save(
        checkpoint,
        path
    )


def load_training_checkpoint(
    path,
    spatial_net,
    planners,
    grand_planner,
    worker,
    target_worker,
    spatial_opt,
    planner_opt,
    grand_opt,
    worker_opt
):

    checkpoint = torch.load(
        path,
        map_location=device,
        weights_only=False
    )

    spatial_net.load_state_dict(
        checkpoint["spatial_net"]
    )

    planners.load_state_dict(
        checkpoint["planners"]
    )

    grand_planner.load_state_dict(
        checkpoint["grand_planner"]
    )

    worker.load_state_dict(
        checkpoint["worker"]
    )

    target_worker.load_state_dict(
        checkpoint["target_worker"]
    )

    spatial_opt.load_state_dict(
        checkpoint["spatial_opt"]
    )

    planner_opt.load_state_dict(
        checkpoint["planner_opt"]
    )

    grand_opt.load_state_dict(
        checkpoint["grand_opt"]
    )

    worker_opt.load_state_dict(
        checkpoint["worker_opt"]
    )

    return checkpoint

# ============================================================
# WITHIN-GAME ARENA ADAPTATION
# ============================================================

class ArenaAdapter:

    def __init__(
        self,
        alpha=0.15
    ):

        self.alpha = alpha

        self.opponent_crop_ema = np.zeros(
            len(CROPS),
            dtype=np.float32
        )

        self.price_ema = np.zeros(
            len(CROPS),
            dtype=np.float32
        )

        self.price_trend = np.zeros(
            len(CROPS),
            dtype=np.float32
        )

        self.initialized = False


    def update(
        self,
        obs
    ):

        p = int(
            obs["player"]
        )

        opponent = obs[
            "farms"
        ][1 - p]

        # ----------------------------------------------------
        # OPPONENT CROP MIX
        # ----------------------------------------------------

        crop_counts = np.zeros(
            len(CROPS),
            dtype=np.float32
        )

        for row in opponent[
            "tiles"
        ]:

            for tile in row:

                if not isinstance(
                    tile,
                    dict
                ):
                    continue

                if tile.get(
                    "kind"
                ) != "PLANT":
                    continue

                crop = tile.get(
                    "crop"
                )

                if crop in CROPS:

                    crop_counts[
                        CROPS.index(crop)
                    ] += 1

        # normalize relative to one quadrant
        crop_counts /= 25.0


        # ----------------------------------------------------
        # MARKET PRICE MOMENTUM
        # ----------------------------------------------------

        prices = (
            obs.get(
                "market", {}
            ).get(
                "prices", {}
            )
        )

        price_vector = np.asarray([
            prices.get(
                crop,
                0
            )
            for crop
            in CROPS
        ], dtype=np.float32)


        if not self.initialized:

            self.opponent_crop_ema = (
                crop_counts
            )

            self.price_ema = (
                price_vector
            )

            self.initialized = True

        else:

            a = self.alpha

            self.opponent_crop_ema = (
                (1 - a)
                * self.opponent_crop_ema
                + a
                * crop_counts
            )

            old_price_ema = (
                self.price_ema.copy()
            )

            self.price_ema = (
                (1 - a)
                * self.price_ema
                + a
                * price_vector
            )

            self.price_trend = (
                self.price_ema
                - old_price_ema
            )


    def adjust_value_maps(
        self,
        value_maps
    ):
        """
        Adjust long-term planting maps according to what the
        opponent is doing NOW.

        Heavy opponent supply:
            lower expected attractiveness.

        Rising market price:
            increase attractiveness.
        """

        adjusted = (
            value_maps.copy()
        )

        for crop_i in range(
            len(CROPS)
        ):

            valid = (
                adjusted[crop_i]
                > INVALID_PLANT_VALUE / 2
            )

            opponent_pressure = (
                self.opponent_crop_ema[
                    crop_i
                ]
            )

            price_momentum = (
                self.price_trend[
                    crop_i
                ]
            )

            # $-equivalent heuristic adjustment
            adjustment = (
                -40.0
                * opponent_pressure
                + 0.50
                * price_momentum
            )

            adjusted[
                crop_i
            ][valid] += adjustment

        return adjusted

# ============================================================
# DETERMINISTIC PLAN FOR CHAMPION / ARENA INFERENCE
# ============================================================

def deterministic_plan(
    out,
    obs,
    plant_logits,
    value_maps,
    cnn_weight=0.85
):

    p = int(
        obs["player"]
    )

    farm = obs[
        "farms"
    ][p]

    hire = int(
        out["hire"]
        .argmax(-1)[0]
        .item()
    )

    land = int(
        out["land"]
        .argmax(-1)[0]
        .item()
    )

    hire, land = (
        apply_expansion_floor(
            obs,
            hire_count=hire,
            buy_land=land
        )
    )

    unlocked = farm.get(
        "unlocked_quadrants",
        ["NW"]
    )

    legal_regions = np.asarray([
        region in unlocked
        for region in REGIONS
    ])

    if not legal_regions.any():
        legal_regions[0] = True


    region_logits = (
        out["regions"][0]
        .detach()
        .cpu()
        .numpy()
    )

    region_logits[
        :,
        ~legal_regions
    ] = -1e9

    regions = (
        region_logits
        .argmax(axis=1)
        .tolist()
    )

    roles = (
        out["roles"][0]
        .argmax(-1)
        .tolist()
    )

    crops = (
        out["crops"][0]
        .argmax(-1)
        .tolist()
    )

    (
        target_x,
        target_y,
        target_values

    ) = select_spatial_targets(

        obs,
        regions,
        roles,
        crops,
        plant_logits,
        value_maps,
        cnn_weight
    )

    return FarmPlan(
        hire_count=hire,
        buy_land=land,
        regions=regions,
        roles=roles,
        crops=crops,
        target_x=target_x,
        target_y=target_y,
        target_values=target_values
    )

# ============================================================
# FROZEN HIERARCHICAL AGENT
#
# Used for:
#   1. champion self-play
#   2. challenger evaluation
#   3. eventual Kaggle arena agent logic
# ============================================================

class FrozenHierarchyAgent:
    """
    Frozen neural policy used for champion self-play and evaluation.

    "Frozen" means its neural-network parameters never receive optimizer
    updates. It can still adapt behavior within a match through ArenaAdapter.

    By default the policy remains STOCHASTIC because PPO was trained by
    sampling from its distributions. Forcing argmax at deployment can turn a
    functional stochastic policy into a poor deterministic policy.
    """

    def __init__(
        self,
        snapshot,
        planner_state_dim,
        grand_state_dim,
        context_dim,
        adaptive=True,
        stochastic_policy=True,
        worker_epsilon=0.05
    ):

        self.spatial_net = (
            SpatialFarmCNN()
            .to(device)
        )

        self.planners = (
            PlannerExperts(
                planner_state_dim,
                hidden=256
            )
            .to(device)
        )

        self.grand_planner = (
            GrandPlanner(
                grand_state_dim
            )
            .to(device)
        )

        self.worker = (
            WorkerTransformer(
                WORKER_TILE_DIM,
                context_dim
            )
            .to(device)
        )

        self.spatial_net.load_state_dict(
            snapshot[
                "spatial_net"
            ]
        )

        self.planners.load_state_dict(
            snapshot[
                "planners"
            ]
        )

        self.grand_planner.load_state_dict(
            snapshot[
                "grand_planner"
            ]
        )

        self.worker.load_state_dict(
            snapshot[
                "worker"
            ]
        )

        # Inference mode. No optimizer is ever attached to these models.
        self.spatial_net.eval()
        self.planners.eval()
        self.grand_planner.eval()
        self.worker.eval()

        for model in (
            self.spatial_net,
            self.planners,
            self.grand_planner,
            self.worker
        ):
            for parameter in model.parameters():
                parameter.requires_grad_(False)

        self.adaptive = bool(
            adaptive
        )

        self.stochastic_policy = bool(
            stochastic_policy
        )

        self.worker_epsilon = float(
            worker_epsilon
        )

        self.reset()


    def reset(
        self
    ):

        self.previous_day = -1

        # Absolute within-game clock.
        self.last_clock = None

        self.phase = 0

        self.plan = None

        self.planner_hidden = [
            self.planners
            .get(i)
            .initial_hidden()

            for i in range(
                NUM_PHASES
            )
        ]

        # Fresh behavioral opponent model for each game.
        # It persists throughout that game.
        self.adapter = (
            ArenaAdapter()
        )


    def __call__(
        self,
        obs,
        configuration=None
    ):

        # ====================================================
        # ROBUST EPISODE RESET
        # ====================================================

        current_clock = (
            int(
                obs["day"]
            )
            * TURNS_PER_DAY

            + int(
                obs["hour"]
            )
        )

        # Reset only when the environment clock goes backwards.
        if (
            self.last_clock is not None
            and
            current_clock < self.last_clock
        ):

            self.reset()

        self.last_clock = (
            current_clock
        )

        # ====================================================
        # WITHIN-MATCH ADAPTATION
        # ====================================================

        if self.adaptive:

            self.adapter.update(
                obs
            )

        # ====================================================
        # DAILY STRATEGIC DECISION
        # ====================================================

        if (
            obs["day"]
            != self.previous_day
        ):

            self.previous_day = (
                obs["day"]
            )

            value_maps = (
                build_long_term_value_maps(
                    obs
                )
            )

            if self.adaptive:

                value_maps = (
                    self.adapter
                    .adjust_value_maps(
                        value_maps
                    )
                )

            (
                spatial_embedding,
                plant_logits
            ) = spatial_forward(
                self.spatial_net,
                obs
            )

            # PPO was trained by sampling. Keep the same policy family in
            # self-play/arena unless deterministic inference is explicitly
            # requested.
            (
                self.phase,
                _,
                _
            ) = choose_phase(
                self.grand_planner,
                obs,
                value_maps,
                deterministic=(
                    not self.stochastic_policy
                )
            )

            planner = (
                self.planners.get(
                    self.phase
                )
            )

            hidden = (
                self.planner_hidden[
                    self.phase
                ]
            )

            state_np = (
                encode_planner_state(
                    obs,
                    spatial_embedding,
                    value_maps
                )
            )

            state = torch.tensor(
                state_np,
                dtype=torch.float32,
                device=device
            )[None]

            with torch.no_grad():

                out = planner(
                    state,
                    hidden
                )

            self.planner_hidden[
                self.phase
            ] = (
                out[
                    "hidden"
                ].detach()
            )

            if self.stochastic_policy:

                (
                    self.plan,
                    _,
                    _,
                    _
                ) = sample_plan_train(
                    out,
                    obs,
                    plant_logits,
                    value_maps,
                    cnn_weight=0.85
                )

            else:

                self.plan = (
                    deterministic_plan(
                        out,
                        obs,
                        plant_logits,
                        value_maps,
                        cnn_weight=0.85
                    )
                )

        # Defensive fallback. Normally the first observation always
        # creates a plan above.
        if self.plan is None:

            value_maps = (
                build_long_term_value_maps(
                    obs
                )
            )

            (
                _,
                plant_logits
            ) = spatial_forward(
                self.spatial_net,
                obs
            )

            self.plan = (
                heuristic_warmup_plan(
                    obs,
                    plant_logits,
                    value_maps
                )
            )

        # ====================================================
        # WORKER EXECUTION
        # ====================================================

        hidden_np = (
            self.planner_hidden[
                self.phase
            ]
            .detach()
            .cpu()
            .numpy()[0]
        )

        tiles, contexts = (
            make_worker_batch(
                obs,
                self.plan,
                hidden_np
            )
        )

        # Small fixed epsilon preserves the DQN policy family used near the
        # end of training without allowing large exploratory randomness.
        actions = (
            select_worker_actions(
                self.worker,
                obs,
                tiles,
                contexts,
                epsilon=(
                    self.worker_epsilon
                )
            )
        )

        return make_kaggle_action(
            obs,
            self.plan,
            actions
        )


# ============================================================
# SELF-PLAY EVALUATION
# ============================================================

def _bad_agent_status(
    status
):
    text = str(
        status
    ).upper()

    return (
        "ERROR" in text
        or
        "INVALID" in text
        or
        "TIMEOUT" in text
    )


def evaluate_challenger(
    challenger_snapshot,
    champion_snapshot,
    planner_state_dim,
    grand_state_dim,
    context_dim,
    num_games=8,
    debug=True
):
    """
    Head-to-head evaluation of two frozen adaptive policies.

    Both policies:
      - use frozen neural weights,
      - retain ArenaAdapter within each match,
      - use the same stochastic deployment policy,
      - alternate player sides.

    Any agent ERROR/INVALID/TIMEOUT is surfaced explicitly.
    """

    challenger_scores = []
    champion_scores = []
    margins = []

    wins = 0
    ties = 0
    error_matches = 0

    for match in range(
        num_games
    ):

        challenger_agent = (
            FrozenHierarchyAgent(
                challenger_snapshot,
                planner_state_dim,
                grand_state_dim,
                context_dim,
                adaptive=True,
                stochastic_policy=True,
                worker_epsilon=0.05
            )
        )

        champion_agent = (
            FrozenHierarchyAgent(
                champion_snapshot,
                planner_state_dim,
                grand_state_dim,
                context_dim,
                adaptive=True,
                stochastic_policy=True,
                worker_epsilon=0.05
            )
        )

        evaluation_env = make(
            "kaggriculture",
            configuration={
                "episodeSteps":
                    TURNS_PER_GAME
            },
            debug=debug
        )

        if match % 2 == 0:

            evaluation_env.run([
                challenger_agent,
                champion_agent
            ])

            challenger_player = 0
            champion_player = 1

        else:

            evaluation_env.run([
                champion_agent,
                challenger_agent
            ])

            challenger_player = 1
            champion_player = 0

        final_states = (
            evaluation_env.steps[
                -1
            ]
        )

        challenger_status = (
            final_states[
                challenger_player
            ].status
        )

        champion_status = (
            final_states[
                champion_player
            ].status
        )

        bad_status = (
            _bad_agent_status(
                challenger_status
            )
            or
            _bad_agent_status(
                champion_status
            )
        )

        if bad_status:

            error_matches += 1

            print(
                f"Eval {match+1:2d}/{num_games} | "
                f"STATUS ERROR | "
                f"challenger={challenger_status} | "
                f"champion={champion_status}"
            )

        final_obs = (
            final_states[
                0
            ].observation
        )

        farms = (
            final_obs[
                "farms"
            ]
        )

        challenger_score = float(
            farms[
                challenger_player
            ][
                "money"
            ]
        )

        champion_score = float(
            farms[
                champion_player
            ][
                "money"
            ]
        )

        margin = (
            challenger_score
            - champion_score
        )

        challenger_scores.append(
            challenger_score
        )

        champion_scores.append(
            champion_score
        )

        margins.append(
            margin
        )

        if (
            not bad_status
            and
            margin > 0
        ):

            wins += 1

        elif (
            not bad_status
            and
            margin == 0
        ):

            ties += 1

        print(
            f"Eval {match+1:2d}/{num_games} | "
            f"Challenger ${challenger_score:,.0f} | "
            f"Champion ${champion_score:,.0f} | "
            f"Margin {margin:+,.0f}"
        )

    win_rate = (
        wins
        / num_games
    )

    tie_rate = (
        ties
        / num_games
    )

    return {
        "wins":
            wins,

        "ties":
            ties,

        "error_matches":
            error_matches,

        "win_rate":
            win_rate,

        "tie_rate":
            tie_rate,

        "mean_margin":
            float(
                np.mean(
                    margins
                )
            ),

        "mean_challenger_score":
            float(
                np.mean(
                    challenger_scores
                )
            ),

        "mean_champion_score":
            float(
                np.mean(
                    champion_scores
                )
            ),

        "challenger_scores":
            challenger_scores,

        "champion_scores":
            champion_scores,

        "margins":
            margins,
    }



In [ ]:
# ============================================================
# CELL 2
# SETUP + TRAINING + SELF-PLAY + GOOGLE DRIVE
# ============================================================

import os
import inspect
import numpy as np
import torch

from collections import Counter
from torch.distributions import Categorical

from google.colab import drive


# ============================================================
# COMPETITION-HORIZON SAFETY CHECK
# ============================================================

assert TURNS_PER_DAY == 24
assert DAYS_PER_GAME == 30
assert TURNS_PER_GAME == 720

print(
    "Kaggriculture horizon: "
    f"{DAYS_PER_GAME} days x {TURNS_PER_DAY} turns/day "
    f"= {TURNS_PER_GAME} turns/game"
)

print(
    "Expansion floor: "
    f"at least {FORCED_MIN_DAILY_HANDS} hands/day; "
    f"force up to {FORCED_MIN_REGIONS} total regions "
    f"through day {FORCED_LAND_LAST_DAY}."
)


# ============================================================
# GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive"
)

# Keep the corrected 30-day run separate from older 300-day checkpoints.
DRIVE_ROOT = (
    "/content/drive/MyDrive/"
    "Kaggriculture_RL_30day_expansion_v2"
)

CHECKPOINT_DIR = (
    f"{DRIVE_ROOT}/checkpoints"
)

CHAMPION_DIR = (
    f"{DRIVE_ROOT}/champions"
)

ARENA_DIR = (
    f"{DRIVE_ROOT}/arena"
)

for folder in [
    DRIVE_ROOT,
    CHECKPOINT_DIR,
    CHAMPION_DIR,
    ARENA_DIR,
]:
    os.makedirs(
        folder,
        exist_ok=True
    )


# Overwritten every 25 games.
LATEST_CHECKPOINT = (
    f"{CHECKPOINT_DIR}/latest.pt"
)

# Only the currently surviving champion is kept.
CHAMPION_LATEST = (
    f"{CHAMPION_DIR}/champion_latest.pt"
)

# Temporary challenger.
# Deleted after each 100-game evaluation.
CHALLENGER_TEMP = (
    f"{CHAMPION_DIR}/challenger_temp.pt"
)


# ============================================================
# TRAINING / SELF-PLAY SETTINGS
# ============================================================

NUM_GAMES = 100000000


# ------------------------------------------------------------
# CURRICULUM
# ------------------------------------------------------------

WORKER_WARMUP_GAMES = 10
GRAND_START_GAME = 30


# ------------------------------------------------------------
# SELF-PLAY LADDER
# ------------------------------------------------------------

SELFPLAY_INTERVAL = 100

SELFPLAY_EVAL_GAMES = 8

# Challenger must win >=58% AND have positive mean margin.
PROMOTION_WIN_RATE = 0.58


PROMOTION_MIN_SCORE = 100.0

# Frozen opponents keep their neural weights fixed, but use the
# same stochastic policy family that PPO/DQN were trained with.
FROZEN_STOCHASTIC_POLICY = True
FROZEN_WORKER_EPSILON = 0.05

# Surface opponent/evaluation exceptions instead of silently
# converting them into mysterious $0 scores.
SELFPLAY_DEBUG = True

# Before accepting/loading a champion, verify that the frozen
# deployment policy can complete games without errors.
CHAMPION_HEALTH_GAMES = 4


# ------------------------------------------------------------
# GOOGLE DRIVE SAVE FREQUENCY
# ------------------------------------------------------------

# Overwrite latest.pt every 25 games.
SAVE_EVERY_GAMES = 25


# ------------------------------------------------------------
# GRAND PLANNER
# ------------------------------------------------------------

PHASE_DECISION_INTERVAL = 10


# ------------------------------------------------------------
# DQN EXPLORATION
#
# Initial training:
#     epsilon decays from 0.60 -> 0.05.
#
# After every 200 COMPLETED games, the ACTIVE LEARNER only is
# reheated to epsilon = 0.25 and then decays back toward 0.05.
#
# Champion opponents are NOT reheated. Their worker epsilon
# remains FROZEN_WORKER_EPSILON above.
# ------------------------------------------------------------

EPS_START = 0.60
EPS_END = 0.05
EPS_DECAY_STEPS = 100_000

EPS_REHEAT = 0.25
EPS_REHEAT_INTERVAL_GAMES = 200
EPS_REHEAT_DECAY_STEPS = 100_000


# ------------------------------------------------------------
# CNN / LONG-TERM VALUE BLEND
# ------------------------------------------------------------

CNN_MAX_WEIGHT = 0.85
CNN_BLEND_STEPS = 150_000


# ------------------------------------------------------------
# WORKER DQN
# ------------------------------------------------------------

TARGET_UPDATE_STEPS = 2000
WORKER_TRAIN_INTERVAL = 4
REPLAY_WARMUP = 1000


# ============================================================
# INITIAL ENVIRONMENT
#
# Used only to obtain a probe observation before networks exist.
# ============================================================

probe_env = make(
    "kaggriculture",
    configuration={
        "episodeSteps":
            TURNS_PER_GAME
    },
    debug=False
)

probe_trainer = (
    probe_env.train([
        None,
        "starter"
    ])
)

probe = (
    probe_trainer.reset()
)


# ============================================================
# SPATIAL CNN
# ============================================================

spatial_net = (
    SpatialFarmCNN()
    .to(device)
)

probe_value_maps = (
    build_long_term_value_maps(
        probe
    )
)

(
    probe_spatial_embedding,
    _
) = spatial_forward(
    spatial_net,
    probe
)


# ============================================================
# NETWORK DIMENSIONS
# ============================================================

planner_state_dim = len(
    encode_planner_state(
        probe,
        probe_spatial_embedding,
        probe_value_maps
    )
)

grand_state_dim = len(
    encode_grand_state(
        probe,
        probe_value_maps
    )
)

HIDDEN = 256


# ============================================================
# PLANNERS
# ============================================================

grand_planner = (
    GrandPlanner(
        grand_state_dim
    )
    .to(device)
)

planners = (
    PlannerExperts(
        planner_state_dim,
        hidden=HIDDEN
    )
    .to(device)
)


# ============================================================
# WORKER TRANSFORMER
# ============================================================

context_dim = (
    HIDDEN
    + len(REGIONS)
    + len(ROLES)
    + len(CROPS)
    + 9
    + 2 * len(CROPS)
)

worker = (
    WorkerTransformer(
        tile_dim=
            WORKER_TILE_DIM,

        context_dim=
            context_dim
    )
    .to(device)
)

target_worker = (
    WorkerTransformer(
        tile_dim=
            WORKER_TILE_DIM,

        context_dim=
            context_dim
    )
    .to(device)
)

target_worker.load_state_dict(
    worker.state_dict()
)


# ============================================================
# OPTIMIZERS
# ============================================================

spatial_opt = torch.optim.Adam(
    spatial_net.parameters(),
    lr=1e-4
)

worker_opt = torch.optim.Adam(
    worker.parameters(),
    lr=5e-4
)

planner_opt = torch.optim.Adam(
    planners.parameters(),
    lr=1e-4
)

grand_opt = torch.optim.Adam(
    grand_planner.parameters(),
    lr=1e-4
)


# ============================================================
# REPLAY BUFFER
#
# We deliberately do NOT save this to Drive because it is
# potentially enormous.
# ============================================================

replay = ReplayBuffer(
    size=150_000
)


# ============================================================
# TRAINING STATE
# ============================================================

START_GAME = 0

global_step = 0

score_history = []

champion_snapshot = None

champion_game = None


# ============================================================
# RESTORE TRAINING CHECKPOINT
# ============================================================

RESUME_TRAINING = True


if (
    RESUME_TRAINING
    and
    os.path.exists(
        LATEST_CHECKPOINT
    )
):

    try:

        checkpoint = (
            load_training_checkpoint(
                LATEST_CHECKPOINT,

                spatial_net,
                planners,
                grand_planner,

                worker,
                target_worker,

                spatial_opt,
                planner_opt,
                grand_opt,
                worker_opt
            )
        )

        START_GAME = (
            checkpoint.get(
                "game",
                -1
            ) + 1
        )

        global_step = (
            checkpoint.get(
                "global_step",
                0
            )
        )

        score_history = (
            checkpoint.get(
                "score_history",
                []
            )
        )

        champion_game = (
            checkpoint.get(
                "champion_game",
                None
            )
        )

        print(
            "\n"
            "========================================\n"
            "TRAINING CHECKPOINT RESTORED\n"
            f"Starting from Game {START_GAME + 1}\n"
            f"Global step: {global_step:,}\n"
            "========================================\n"
        )

    except Exception as e:

        print(
            "\nExisting checkpoint is not compatible "
            "with the current architecture."
        )

        print(
            "Starting with fresh weights."
        )

        print(
            f"Checkpoint error: {e}\n"
        )

        START_GAME = 0

        global_step = 0

        score_history = []


# ============================================================
# RESTORE CURRENT CHAMPION
# ============================================================

if os.path.exists(
    CHAMPION_LATEST
):

    try:

        champion_snapshot = (
            load_policy_snapshot(
                CHAMPION_LATEST
            )
        )

        champion_game = (
            champion_snapshot.get(
                "game",
                champion_game
            )
        )

        print(
            "\n"
            "========================================\n"
            f"CHAMPION LOADED: Game {champion_game}\n"
            "Weights are frozen during self-play.\n"
            "Opponent adaptation remains enabled.\n"
            "========================================\n"
        )

    except Exception as e:

        print(
            "Existing champion is incompatible "
            "with the current architecture."
        )

        print(
            f"Champion load error: {e}"
        )

        champion_snapshot = None

        champion_game = None


# ============================================================
# CREATE TRAINING ENVIRONMENT
#
# Before first champion:
#     learner vs starter
#
# After first champion:
#     learner vs frozen adaptive champion
# ============================================================

def make_training_trainer(
    champion
):

    training_env = make(
        "kaggriculture",
        configuration={
            "episodeSteps":
                TURNS_PER_GAME
        },
        debug=(
            SELFPLAY_DEBUG
            if champion is not None
            else False
        )
    )


    if champion is None:

        opponent = "starter"

        opponent_name = (
            "starter"
        )

    else:

        # IMPORTANT:
        #
        # Neural weights are frozen because this object loads a
        # snapshot and is never given an optimizer.
        #
        # adaptive=True means ArenaAdapter still observes the
        # current opponent and modifies strategic value maps.
        opponent = FrozenHierarchyAgent(
            champion,

            planner_state_dim,
            grand_state_dim,
            context_dim,

            adaptive=True,

            stochastic_policy=
                FROZEN_STOCHASTIC_POLICY,

            worker_epsilon=
                FROZEN_WORKER_EPSILON
        )

        opponent_name = (
            f"Champion "
            f"{champion.get('game', '?')}"
        )


    training_trainer = (
        training_env.train([
            None,
            opponent
        ])
    )


    return (
        training_env,
        training_trainer,
        opponent_name
    )



# ============================================================
# SELF-PLAY HEALTH CHECKS
# ============================================================

def status_is_bad(
    status
):
    text = str(
        status
    ).upper()

    return (
        "ERROR" in text
        or
        "INVALID" in text
        or
        "TIMEOUT" in text
    )


def validate_snapshot_vs_starter(
    snapshot,
    num_games=CHAMPION_HEALTH_GAMES
):
    """
    Verifies that a frozen snapshot actually functions when deployed.

    This catches exactly the failure mode that produced a champion
    which appeared in the ladder but scored $0 because its callable
    policy was erroring or collapsing after freezing.
    """

    scores = []
    errors = 0

    for match in range(
        num_games
    ):

        candidate = (
            FrozenHierarchyAgent(
                snapshot,

                planner_state_dim,
                grand_state_dim,
                context_dim,

                adaptive=True,

                stochastic_policy=
                    FROZEN_STOCHASTIC_POLICY,

                worker_epsilon=
                    FROZEN_WORKER_EPSILON
            )
        )

        health_env = make(
            "kaggriculture",
            configuration={
                "episodeSteps":
                    TURNS_PER_GAME
            },
            debug=SELFPLAY_DEBUG
        )

        if match % 2 == 0:

            health_env.run([
                candidate,
                "starter"
            ])

            candidate_player = 0

        else:

            health_env.run([
                "starter",
                candidate
            ])

            candidate_player = 1

        final_states = (
            health_env.steps[
                -1
            ]
        )

        candidate_status = (
            final_states[
                candidate_player
            ].status
        )

        if status_is_bad(
            candidate_status
        ):

            errors += 1

        final_obs = (
            final_states[
                0
            ].observation
        )

        score = float(
            final_obs[
                "farms"
            ][
                candidate_player
            ][
                "money"
            ]
        )

        scores.append(
            score
        )

        print(
            f"Champion health "
            f"{match+1}/{num_games} | "
            f"score=${score:,.0f} | "
            f"status={candidate_status}"
        )

    return {
        "scores":
            scores,

        "mean_score":
            float(
                np.mean(
                    scores
                )
            ),

        "error_matches":
            errors,

        "healthy":
            (
                errors == 0
                and
                float(
                    np.mean(
                        scores
                    )
                ) >= PROMOTION_MIN_SCORE
            ),
    }


# ============================================================
# VERIFY CELL 1 COMPATIBILITY
# ============================================================

frozen_signature = (
    inspect.signature(
        FrozenHierarchyAgent.__init__
    )
)

required_frozen_args = {
    "stochastic_policy",
    "worker_epsilon",
}

missing_frozen_args = (
    required_frozen_args
    - set(
        frozen_signature.parameters
    )
)

if missing_frozen_args:

    raise RuntimeError(
        "Cell 1 is stale/incompatible. "
        "Rerun the corrected Cell 1 before Cell 2. "
        f"Missing FrozenHierarchyAgent args: "
        f"{sorted(missing_frozen_args)}"
    )


# ============================================================
# VERIFY A LOADED CHAMPION BEFORE TRAINING AGAINST IT
# ============================================================

if champion_snapshot is not None:

    print(
        "\nValidating loaded champion "
        f"{champion_game} before self-play..."
    )

    loaded_champion_health = (
        validate_snapshot_vs_starter(
            champion_snapshot
        )
    )

    if not loaded_champion_health[
        "healthy"
    ]:

        print(
            "\nWARNING: saved champion failed deployment "
            "health check.\n"
            f"Mean score: "
            f"${loaded_champion_health['mean_score']:,.0f}\n"
            f"Error matches: "
            f"{loaded_champion_health['error_matches']}\n"
            "It will NOT be used as the training opponent. "
            "The saved file is left untouched for diagnosis.\n"
        )

        champion_snapshot = None
        champion_game = None


env, trainer, opponent_name = (
    make_training_trainer(
        champion_snapshot
    )
)


print(
    f"Current training opponent: "
    f"{opponent_name}"
)


# ============================================================
# TRAIN ACROSS GAMES
# ============================================================

for game in range(
    START_GAME,
    NUM_GAMES
):


    # ========================================================
    # PERIODIC EXPLORATION REHEAT
    #
    # Zero-based `game == 200` means Game 201 is about to start:
    # exactly 200 games have been completed. The learner keeps
    # ALL weights, optimizer state, replay memory, and training
    # history. Only worker epsilon is reheated.
    # ========================================================

    if (
        game > 0
        and
        game % EPS_REHEAT_INTERVAL_GAMES == 0
    ):

        print(
            "\n"
            "############################################################\n"
            "EXPLORATION REHEAT\n"
            f"Starting Game {game + 1} after "
            f"{game} completed games.\n"
            f"Active learner epsilon resets to {EPS_REHEAT:.2f} "
            f"and will decay back toward {EPS_END:.2f}.\n"
            "Learner weights are NOT reset.\n"
            "Champion weights/epsilon are NOT changed.\n"
            "############################################################\n"
        )


    # ========================================================
    # CURRICULUM
    # ========================================================

    worker_warmup = (
        game
        < WORKER_WARMUP_GAMES
    )

    train_planner_now = (
        game
        >= WORKER_WARMUP_GAMES
    )

    train_grand_now = (
        game
        >= GRAND_START_GAME
    )


    if worker_warmup:

        training_stage = (
            "SPATIAL + WORKER WARMUP"
        )

    elif not train_grand_now:

        training_stage = (
            "PLANNER TRAINING"
        )

    else:

        training_stage = (
            "FULL HIERARCHY"
        )


    # ========================================================
    # INITIALIZE PLANNER EXPERTS AT GRAND START
    # ========================================================

    if game == GRAND_START_GAME:

        base_state = (
            planners
            .get(0)
            .state_dict()
        )

        for phase_i in range(
            1,
            NUM_PHASES
        ):

            planners.get(
                phase_i
            ).load_state_dict(
                base_state
            )

        print(
            "\nCopied Planner 0 into "
            "the other planner experts.\n"
        )


    # ========================================================
    # NEW GAME
    # ========================================================

    obs = (
        trainer.reset()
    )


    # --------------------------------------------------------
    # ADAPTIVE OPPONENT MODEL
    #
    # This resets between games but learns about the current
    # opponent continuously DURING the game.
    # --------------------------------------------------------

    arena_adapter = (
        ArenaAdapter()
    )


    planner_hidden = [

        planners
        .get(i)
        .initial_hidden()

        for i in range(
            NUM_PHASES
        )
    ]


    previous_day = -1

    previous_shops = None

    phase = 0

    plan = None

    current_record = None

    current_grand_record = None


    planner_trajs = {
        i: []
        for i in range(
            NUM_PHASES
        )
    }

    grand_traj = []


    # --------------------------------------------------------
    # LOSSES
    # --------------------------------------------------------

    worker_losses = []

    planner_losses = []

    spatial_losses = []

    grand_loss_value = np.nan


    # --------------------------------------------------------
    # DIAGNOSTICS
    # --------------------------------------------------------

    action_counts = Counter()

    market_counts = Counter()

    max_money = 0.0

    max_strategic_value = (
        strategic_value(
            obs
        )
    )


    done = False

    step = 0


    # ========================================================
    # PLAY ONE GAME
    # ========================================================

    while (
        not done
        and
        step < TURNS_PER_GAME
    ):


        # ====================================================
        # ADAPT TO OPPONENT EVERY TURN
        # ====================================================

        arena_adapter.update(
            obs
        )


        # ====================================================
        # NEW DAY
        # ====================================================

        if (
            obs["day"]
            != previous_day
        ):


            # =================================================
            # FINISH PREVIOUS DAILY PLANNER DECISION
            # =================================================

            if (
                current_record
                is not None
            ):

                current_record[
                    "reward"
                ] = (

                    strategic_value(
                        obs
                    )

                    - current_record[
                        "start_value"
                    ]

                ) / 100.0


                planner_trajs[
                    current_record[
                        "phase"
                    ]
                ].append(
                    current_record
                )


                current_record = None


            previous_day = (
                obs["day"]
            )


            # =================================================
            # BASE LONG-TERM ECONOMIC MAP
            # =================================================

            base_value_maps = (
                build_long_term_value_maps(
                    obs
                )
            )


            # =================================================
            # ADAPT VALUE MAP TO CURRENT OPPONENT
            #
            # Neural weights are NOT being changed here.
            # Strategy is changing from live opponent evidence.
            # =================================================

            value_maps = (
                arena_adapter
                .adjust_value_maps(
                    base_value_maps
                )
            )


            # =================================================
            # TRAIN SPATIAL CNN
            #
            # Train on the ADAPTED map so the learner experiences
            # opponent-aware spatial decisions during training.
            # =================================================

            spatial_loss = (
                train_spatial_model(
                    spatial_net,
                    spatial_opt,
                    obs,
                    value_maps
                )
            )


            if (
                spatial_loss
                is not None
            ):

                spatial_losses.append(
                    spatial_loss
                )


            # Recompute spatial inference after update.
            (
                spatial_embedding,
                plant_logits

            ) = spatial_forward(
                spatial_net,
                obs
            )


            # =================================================
            # CNN / HEURISTIC BLEND
            # =================================================

            cnn_weight = min(

                CNN_MAX_WEIGHT,

                CNN_MAX_WEIGHT
                * (
                    global_step
                    / CNN_BLEND_STEPS
                )
            )


            # =================================================
            # GRAND PLANNER
            # =================================================

            current_shops = tuple(
                obs.get(
                    "town", {}
                ).get(
                    "unlocked_shops",
                    []
                )
            )


            shops_changed = (

                previous_shops
                is not None

                and

                current_shops
                != previous_shops
            )


            phase_checkpoint = (

                obs["day"]
                % PHASE_DECISION_INTERVAL
                == 0
            )


            replan_phase = (

                phase is None

                or

                shops_changed

                or

                phase_checkpoint
            )


            old_phase = (
                phase
            )


            # ------------------------------------------------
            # BEFORE GRAND TRAINING
            # ------------------------------------------------

            if not train_grand_now:

                phase = 0


            # ------------------------------------------------
            # GRAND PLANNER ACTIVE
            # ------------------------------------------------

            elif replan_phase:


                if (
                    current_grand_record
                    is not None
                ):

                    current_grand_record[
                        "reward"
                    ] = (

                        strategic_value(
                            obs
                        )

                        - current_grand_record[
                            "start_value"
                        ]

                    ) / 100.0


                    grand_traj.append(
                        current_grand_record
                    )


                grand_state_np = (
                    encode_grand_state(
                        obs,
                        value_maps
                    )
                )


                with torch.no_grad():

                    (
                        phase,
                        phase_logp,
                        phase_value

                    ) = choose_phase(
                        grand_planner,
                        obs,
                        value_maps
                    )


                current_grand_record = {

                    "state":
                        grand_state_np,

                    "phase":
                        phase,

                    "logp":
                        float(
                            phase_logp.item()
                        ),

                    "value":
                        float(
                            phase_value.item()
                        ),

                    "start_value":
                        strategic_value(
                            obs
                        ),
                }


            previous_shops = (
                current_shops
            )


            # =================================================
            # DAILY STATUS
            # =================================================

            phase_text = ""


            if (
                old_phase
                is not None
                and
                old_phase != phase
            ):

                phase_text = (
                    f" | SWITCH "
                    f"{old_phase}→{phase}"
                )


            crop_values = (
                crop_value_features(
                    value_maps
                )
            )


            best_crop_i = int(
                np.argmax(
                    crop_values[
                        ::2
                    ]
                )
            )


            print(

                f"Game {game+1:4d} | "

                f"Day "
                f"{obs['day']:3d}/"
                f"{DAYS_PER_GAME-1} | "

                f"Turn "
                f"{step:4d}/"
                f"{TURNS_PER_GAME} | "

                f"Phase {phase} | "

                f"Best "
                f"{CROPS[best_crop_i]} | "

                f"CNN "
                f"{cnn_weight:.2f} | "

                f"vs {opponent_name}"

                f"{phase_text}"
            )


            # =================================================
            # WORKER WARMUP
            # =================================================

            if worker_warmup:

                plan = (
                    heuristic_warmup_plan(
                        obs,
                        plant_logits,
                        value_maps
                    )
                )


                hidden_np = (
                    planner_hidden[
                        0
                    ]
                    .detach()
                    .cpu()
                    .numpy()[0]
                )


            # =================================================
            # TRAINED PHASE PLANNER
            # =================================================

            else:

                planner = (
                    planners.get(
                        phase
                    )
                )


                hidden = (
                    planner_hidden[
                        phase
                    ]
                )


                hidden_in = (
                    hidden
                    .detach()
                    .cpu()
                    .numpy()[0]
                )


                state_np = (
                    encode_planner_state(
                        obs,
                        spatial_embedding,
                        value_maps
                    )
                )


                state = torch.tensor(
                    state_np,
                    dtype=torch.float32,
                    device=device
                )[None]


                with torch.no_grad():

                    out = planner(
                        state,
                        hidden
                    )


                planner_hidden[
                    phase
                ] = (
                    out[
                        "hidden"
                    ]
                    .detach()
                )


                (
                    plan,
                    planner_action,
                    planner_logp,
                    planner_value

                ) = sample_plan_train(

                    out,
                    obs,
                    plant_logits,
                    value_maps,
                    cnn_weight
                )


                current_record = {

                    "phase":
                        phase,

                    "state":
                        state_np,

                    "hidden":
                        hidden_in,

                    "action":
                        planner_action,

                    "logp":
                        planner_logp,

                    "value":
                        planner_value,

                    "start_value":
                        strategic_value(
                            obs
                        ),
                }


                hidden_np = (
                    planner_hidden[
                        phase
                    ]
                    .detach()
                    .cpu()
                    .numpy()[0]
                )


        # ====================================================
        # WORKER TRANSFORMER
        # ====================================================

        if worker_warmup:

            hidden_np = (
                planner_hidden[
                    0
                ]
                .detach()
                .cpu()
                .numpy()[0]
            )

        else:

            hidden_np = (
                planner_hidden[
                    phase
                ]
                .detach()
                .cpu()
                .numpy()[0]
            )


        tiles, contexts = (
            make_worker_batch(
                obs,
                plan,
                hidden_np
            )
        )


        # ====================================================
        # EPSILON WITH PERIODIC REHEATING
        # ====================================================

        if game < EPS_REHEAT_INTERVAL_GAMES:

            # Initial exploration schedule.
            epsilon = (

                EPS_END

                + (
                    EPS_START
                    - EPS_END
                )

                * np.exp(
                    -global_step
                    / EPS_DECAY_STEPS
                )
            )

            exploration_mode = (
                "INITIAL DECAY"
            )

            exploration_cycle_game = (
                game
            )

        else:

            # Reheat after Games 200, 400, 600, ...
            #
            # We calculate position within the current 200-game
            # exploration cycle from the game/turn indices. This
            # means the schedule reconstructs automatically after
            # a Colab checkpoint resume; no extra checkpoint field
            # is required.
            exploration_cycle = (
                game
                // EPS_REHEAT_INTERVAL_GAMES
            )

            exploration_cycle_game = (
                game
                % EPS_REHEAT_INTERVAL_GAMES
            )

            approximate_steps_since_reheat = (
                exploration_cycle_game
                * TURNS_PER_GAME
                + step
            )

            epsilon = (

                EPS_END

                + (
                    EPS_REHEAT
                    - EPS_END
                )

                * np.exp(
                    -approximate_steps_since_reheat
                    / EPS_REHEAT_DECAY_STEPS
                )
            )

            exploration_mode = (
                f"REHEAT #{exploration_cycle}"
            )


        # ====================================================
        # WORKER ACTIONS
        # ====================================================

        worker_actions = (
            select_worker_actions(
                worker,
                obs,
                tiles,
                contexts,
                epsilon
            )
        )


        for action_i in (
            worker_actions
        ):

            op, arg = (
                WORKER_ACTIONS[
                    action_i
                ]
            )


            if arg is None:

                action_counts[
                    op
                ] += 1

            else:

                action_counts[
                    f"{op}_{arg}"
                ] += 1


        action = (
            make_kaggle_action(
                obs,
                plan,
                worker_actions
            )
        )


        for market_order in (
            action["market"]
        ):

            if market_order:

                market_counts[
                    market_order[0]
                ] += 1


        # ====================================================
        # ENVIRONMENT STEP
        # ====================================================

        old_day = (
            obs["day"]
        )


        next_obs, _, done, _ = (
            trainer.step(
                action
            )
        )


        option_finished = (

            done

            or

            next_obs[
                "day"
            ] != old_day
        )


        # ====================================================
        # NEXT WORKERS
        # ====================================================

        p_next = int(
            next_obs[
                "player"
            ]
        )


        next_workers = min(

            MAX_WORKERS,

            1
            + len(
                next_obs[
                    "farms"
                ][
                    p_next
                ].get(
                    "hands",
                    []
                )
            )
        )


        # ====================================================
        # STORE WORKER TRANSITIONS
        # ====================================================

        for (
            worker_idx,
            worker_action

        ) in enumerate(
            worker_actions
        ):


            reward_i = (
                worker_reward(
                    obs,
                    next_obs,
                    plan,
                    worker_idx,
                    worker_action
                )
            )


            if (
                not option_finished
                and
                worker_idx
                < next_workers
            ):

                next_tiles = (
                    encode_worker_region(
                        next_obs,
                        plan,
                        worker_idx
                    )
                )


                next_context = (
                    build_worker_context(
                        next_obs,
                        plan,
                        worker_idx,
                        hidden_np
                    )
                )


                next_mask = (
                    worker_mask(
                        next_obs,
                        worker_idx
                    )
                )


            else:

                next_tiles = (
                    np.zeros_like(
                        tiles[
                            worker_idx
                        ]
                    )
                )


                next_context = (
                    np.zeros_like(
                        contexts[
                            worker_idx
                        ]
                    )
                )


                next_mask = (
                    np.zeros(
                        len(
                            WORKER_ACTIONS
                        ),
                        dtype=bool
                    )
                )


                next_mask[0] = True


            replay.add(

                tiles[
                    worker_idx
                ],

                contexts[
                    worker_idx
                ],

                worker_action,

                reward_i,

                next_tiles,

                next_context,

                option_finished,

                next_mask
            )


        # ====================================================
        # ADVANCE TURN
        # ====================================================

        obs = next_obs

        step += 1

        global_step += 1


        # ====================================================
        # METRICS
        # ====================================================

        p_now = int(
            obs["player"]
        )


        max_money = max(

            max_money,

            float(
                obs[
                    "farms"
                ][
                    p_now
                ][
                    "money"
                ]
            )
        )


        max_strategic_value = max(

            max_strategic_value,

            strategic_value(
                obs
            )
        )


        # ====================================================
        # ONLINE WORKER TRAINING
        # ====================================================

        if (
            len(replay)
            >= REPLAY_WARMUP

            and

            global_step
            % WORKER_TRAIN_INTERVAL
            == 0
        ):

            worker_loss = (
                train_worker(
                    worker,
                    target_worker,
                    worker_opt,
                    replay
                )
            )


            if (
                worker_loss
                is not None
            ):

                worker_losses.append(
                    worker_loss
                )


        # ====================================================
        # TARGET NETWORK UPDATE
        # ====================================================

        if (
            global_step
            % TARGET_UPDATE_STEPS
            == 0
        ):

            target_worker.load_state_dict(
                worker.state_dict()
            )


    # ========================================================
    # GAME FINISHED
    # ========================================================

    # --------------------------------------------------------
    # DO NOT ALLOW SILENTLY BROKEN CHAMPIONS
    # --------------------------------------------------------

    if champion_snapshot is not None:

        try:

            opponent_status = (
                env.steps[
                    -1
                ][1].status
            )

            if status_is_bad(
                opponent_status
            ):

                raise RuntimeError(
                    f"Frozen Champion {champion_game} "
                    f"failed during Game {game+1}. "
                    f"Opponent status: {opponent_status}. "
                    "Because SELFPLAY_DEBUG=True, inspect the "
                    "traceback printed above rather than treating "
                    "the resulting $0 as a valid score."
                )

        except IndexError:

            raise RuntimeError(
                "Could not inspect opponent status at end of game. "
                "The training environment did not contain both agents."
            )


    p = int(
        obs["player"]
    )


    my_score = float(
        obs[
            "farms"
        ][p][
            "money"
        ]
    )


    opponent_score = float(
        obs[
            "farms"
        ][
            1 - p
        ][
            "money"
        ]
    )


    margin = (
        my_score
        - opponent_score
    )


    game_reward = (

        10.0

        * np.tanh(
            margin
            / 5000.0
        )
    )


    score_history.append(
        margin
    )


    # ========================================================
    # FINISH LAST DAILY PLANNER RECORD
    # ========================================================

    if (
        current_record
        is not None
    ):

        current_record[
            "reward"
        ] = (

            strategic_value(
                obs
            )

            - current_record[
                "start_value"
            ]

        ) / 100.0


        planner_trajs[
            current_record[
                "phase"
            ]
        ].append(
            current_record
        )


    # Final competitive reward to every planner expert used.
    for trajectory in (
        planner_trajs.values()
    ):

        if trajectory:

            trajectory[
                -1
            ][
                "reward"
            ] += game_reward


    # ========================================================
    # FINISH LAST GRAND SEGMENT
    # ========================================================

    if (
        current_grand_record
        is not None
    ):

        current_grand_record[
            "reward"
        ] = (

            strategic_value(
                obs
            )

            - current_grand_record[
                "start_value"
            ]

        ) / 100.0


        current_grand_record[
            "reward"
        ] += game_reward


        grand_traj.append(
            current_grand_record
        )


    # ========================================================
    # TRAIN PHASE PLANNERS
    # ========================================================

    if train_planner_now:

        for (
            phase_id,
            trajectory

        ) in planner_trajs.items():


            if (
                len(
                    trajectory
                ) < 2
            ):

                continue


            planner_loss = (
                train_planner(
                    planners.get(
                        phase_id
                    ),
                    planner_opt,
                    trajectory,
                    gamma=.995
                )
            )


            if (
                planner_loss
                is not None
            ):

                planner_losses.append(
                    planner_loss
                )


    # ========================================================
    # TRAIN GRAND PLANNER
    # ========================================================

    if (
        train_grand_now

        and

        len(
            grand_traj
        ) >= 2
    ):


        grand_states = (
            torch.tensor(
                np.stack([
                    record[
                        "state"
                    ]
                    for record
                    in grand_traj
                ]),
                dtype=torch.float32,
                device=device
            )
        )


        grand_actions = (
            torch.tensor(
                [
                    record[
                        "phase"
                    ]
                    for record
                    in grand_traj
                ],
                dtype=torch.long,
                device=device
            )
        )


        grand_old_logp = (
            torch.tensor(
                [
                    record[
                        "logp"
                    ]
                    for record
                    in grand_traj
                ],
                dtype=torch.float32,
                device=device
            )
        )


        # ----------------------------------------------------
        # DISCOUNTED GRAND RETURNS
        # ----------------------------------------------------

        returns = []

        R = 0.0


        for record in reversed(
            grand_traj
        ):

            R = (

                record[
                    "reward"
                ]

                + .99 * R
            )

            returns.append(
                R
            )


        returns.reverse()


        grand_returns = (
            torch.tensor(
                returns,
                dtype=torch.float32,
                device=device
            )
        )


        # ----------------------------------------------------
        # PPO
        # ----------------------------------------------------

        for _ in range(4):


            out = (
                grand_planner(
                    grand_states
                )
            )


            dist = Categorical(
                logits=out[
                    "phase_logits"
                ]
            )


            new_logp = (
                dist.log_prob(
                    grand_actions
                )
            )


            values = (
                out[
                    "value"
                ].squeeze(1)
            )


            advantage = (

                grand_returns

                - values.detach()
            )


            advantage = (

                advantage
                - advantage.mean()

            ) / (

                advantage.std(
                    unbiased=False
                )

                + 1e-8
            )


            ratio = torch.exp(
                new_logp
                - grand_old_logp
            )


            clipped_ratio = (
                torch.clamp(
                    ratio,
                    .8,
                    1.2
                )
            )


            policy_loss = -torch.min(

                ratio
                * advantage,

                clipped_ratio
                * advantage

            ).mean()


            value_loss = (
                F.mse_loss(
                    values,
                    grand_returns
                )
            )


            entropy = (
                dist.entropy()
                .mean()
            )


            grand_loss = (

                policy_loss

                + .5
                * value_loss

                - .01
                * entropy
            )


            grand_opt.zero_grad()

            grand_loss.backward()


            torch.nn.utils.clip_grad_norm_(
                grand_planner.parameters(),
                10
            )


            grand_opt.step()


        grand_loss_value = (
            float(
                grand_loss.item()
            )
        )


    # ========================================================
    # REPORT
    # ========================================================

    avg_margin = float(
        np.mean(
            score_history[
                -20:
            ]
        )
    )


    win_rate = float(
        np.mean([
            result > 0

            for result
            in score_history[
                -20:
            ]
        ])
    )


    worker_loss_value = (

        float(
            np.mean(
                worker_losses
            )
        )

        if worker_losses

        else np.nan
    )


    planner_loss_value = (

        float(
            np.mean(
                planner_losses
            )
        )

        if planner_losses

        else np.nan
    )


    spatial_loss_value = (

        float(
            np.mean(
                spatial_losses
            )
        )

        if spatial_losses

        else np.nan
    )


    print(

        "\n"
        "============================================================\n"

        f"GAME {game+1} COMPLETE | "
        f"{training_stage}\n"

        f"Opponent:            "
        f"{opponent_name}\n\n"

        f"Our score:           "
        f"${my_score:,.0f}\n"

        f"Opponent score:      "
        f"${opponent_score:,.0f}\n"

        f"Margin:              "
        f"{margin:+,.0f}\n"

        f"Game reward:         "
        f"{game_reward:+.3f}\n\n"

        f"20-game avg margin:  "
        f"{avg_margin:+,.0f}\n"

        f"20-game win rate:    "
        f"{win_rate:.0%}\n\n"

        f"Max money:           "
        f"${max_money:,.0f}\n"

        f"Max strategic value: "
        f"${max_strategic_value:,.0f}\n\n"

        f"Epsilon:             "
        f"{epsilon:.3f}\n"

        f"Exploration mode:    "
        f"{exploration_mode}\n"

        f"Cycle game:          "
        f"{exploration_cycle_game + 1}/"
        f"{EPS_REHEAT_INTERVAL_GAMES}\n"

        f"CNN target weight:   "
        f"{cnn_weight:.3f}\n"

        f"Replay size:         "
        f"{len(replay):,}\n\n"

        f"Spatial CNN loss:    "
        f"{spatial_loss_value:.4f}\n"

        f"Worker loss:         "
        f"{worker_loss_value:.4f}\n"

        f"Planner loss:        "
        f"{planner_loss_value:.4f}\n"

        f"Grand loss:          "
        f"{grand_loss_value:.4f}\n\n"

        "ACTIONS\n"

        f"Plant:   "
        f"{sum(
            action_counts[
                f'PLANT_{crop}'
            ]
            for crop
            in CROPS
        ):,}\n"

        f"Water:   "
        f"{action_counts['WATER']:,}\n"

        f"Harvest: "
        f"{action_counts['HARVEST']:,}\n"

        f"Place:   "
        f"{action_counts['PLACE']:,}\n"

        f"Dig:     "
        f"{action_counts['DIG']:,}\n\n"

        "MARKET\n"

        f"BuySeed: "
        f"{market_counts['BUY_SEED']:,}\n"

        f"Sell:    "
        f"{market_counts['SELL']:,}\n"

        f"Hire:    "
        f"{market_counts['HIRE']:,}\n"

        f"BuyLand: "
        f"{market_counts['BUY_LAND']:,}\n"

        "============================================================\n"
    )


    # ========================================================
    # SELF-PLAY LADDER — EVERY 100 GAMES
    # ========================================================

    if (
        (game + 1)
        % SELFPLAY_INTERVAL
        == 0
    ):


        checkpoint_game = (
            game + 1
        )


        print(
            "\n"
            "############################################################\n"
            f"SELF-PLAY CHECKPOINT — GAME {checkpoint_game}\n"
            "############################################################"
        )


        # ----------------------------------------------------
        # FREEZE CURRENT LEARNER AS TEMPORARY CHALLENGER
        # ----------------------------------------------------

        challenger_snapshot = (
            make_policy_snapshot(
                spatial_net,
                planners,
                grand_planner,
                worker,

                game=
                    checkpoint_game,

                global_step=
                    global_step
            )
        )


        save_policy_snapshot(
            CHALLENGER_TEMP,
            challenger_snapshot
        )


        # ----------------------------------------------------
        # NO EXISTING CHAMPION:
        # GAME 100 AUTOMATICALLY BECOMES FIRST CHAMPION
        # ----------------------------------------------------

        if (
            champion_snapshot
            is None
        ):

            print(
                "\nNo champion currently exists. "
                "Validating challenger against starter..."
            )

            candidate_health = (
                validate_snapshot_vs_starter(
                    challenger_snapshot
                )
            )

            if candidate_health[
                "healthy"
            ]:

                champion_snapshot = (
                    challenger_snapshot
                )

                champion_game = (
                    checkpoint_game
                )

                save_policy_snapshot(
                    CHAMPION_LATEST,
                    champion_snapshot
                )

                print(
                    "\nINITIAL CHAMPION CREATED\n"
                    f"Champion = Game "
                    f"{champion_game}\n"
                    f"Frozen deployment mean score: "
                    f"${candidate_health['mean_score']:,.0f}\n"
                    "Weights are frozen; ArenaAdapter remains live.\n"
                )

            else:

                print(
                    "\nINITIAL CHAMPION REJECTED\n"
                    f"Candidate mean score: "
                    f"${candidate_health['mean_score']:,.0f}\n"
                    f"Error matches: "
                    f"{candidate_health['error_matches']}\n"
                    "Training will continue against starter until "
                    "the next 100-game checkpoint.\n"
                )


        # ----------------------------------------------------
        # CHALLENGER VS CURRENT CHAMPION
        # ----------------------------------------------------

        else:


            print(
                f"\nChallenger "
                f"{checkpoint_game} "
                f"vs Champion "
                f"{champion_game}"
            )


            result = (
                evaluate_challenger(

                    challenger_snapshot,

                    champion_snapshot,

                    planner_state_dim,

                    grand_state_dim,

                    context_dim,

                    num_games=
                        SELFPLAY_EVAL_GAMES,

                    debug=
                        SELFPLAY_DEBUG
                )
            )


            print(
                "\n"
                "============================================================\n"
                f"CHALLENGER {checkpoint_game} "
                f"vs CHAMPION {champion_game}\n"
                "============================================================\n"

                f"Challenger wins:      "
                f"{result['wins']}/{SELFPLAY_EVAL_GAMES}\n"

                f"Challenger win rate:  "
                f"{result['win_rate']:.1%}\n"

                f"Challenger avg score: "
                f"${result['mean_challenger_score']:,.0f}\n"

                f"Champion avg score:   "
                f"${result['mean_champion_score']:,.0f}\n"

                f"Mean margin:           "
                f"{result['mean_margin']:+,.0f}\n"

                f"Error matches:         "
                f"{result['error_matches']}\n"

                f"Margins:               "
                f"{result['margins']}\n"

                "============================================================"
            )


            promoted = (

                # Must beat champion consistently.
                result[
                    "win_rate"
                ] >= PROMOTION_WIN_RATE

                and

                # Must actually outperform it economically.
                result[
                    "mean_margin"
                ] > 0

                and

                # Avoid promoting two totally broken agents
                # simply because one ended at $1 instead of $0.
                result[
                    "mean_challenger_score"
                ] >= PROMOTION_MIN_SCORE

                and

                result[
                    "error_matches"
                ] == 0
            )


            # =================================================
            # CHALLENGER WINS
            # =================================================

            if promoted:

                old_champion_game = (
                    champion_game
                )


                # The snapshot is already a completely detached
                # copy of the learner's weights.
                #
                # From this point onward these neural weights are
                # frozen when used as the opponent.
                champion_snapshot = (
                    challenger_snapshot
                )


                champion_game = (
                    checkpoint_game
                )


                # Overwrite old champion to save Drive space.
                save_policy_snapshot(
                    CHAMPION_LATEST,
                    champion_snapshot
                )


                print(
                    "\n"
                    "**************** PROMOTED ****************\n"
                    f"Old Champion: Game {old_champion_game}\n"
                    f"New Champion: Game {champion_game}\n"
                    "Neural weights are FROZEN.\n"
                    "ArenaAdapter remains ACTIVE, so behavior still\n"
                    "adapts to the current opponent within each game.\n"
                    "******************************************\n"
                )


            # =================================================
            # CHALLENGER LOSES
            # =================================================

            else:


                print(
                    "\n"
                    "**************** REJECTED ****************\n"
                    f"Champion {champion_game} remains.\n"
                    "The failed challenger will not be retained.\n"
                    "******************************************\n"
                )


        # ----------------------------------------------------
        # DELETE TEMPORARY CHALLENGER COPY
        # ----------------------------------------------------

        if os.path.exists(
            CHALLENGER_TEMP
        ):

            os.remove(
                CHALLENGER_TEMP
            )


        # ----------------------------------------------------
        # REBUILD TRAINER AGAINST SURVIVING CHAMPION
        #
        # Frozen weights + live ArenaAdapter
        # ----------------------------------------------------

        env, trainer, opponent_name = (
            make_training_trainer(
                champion_snapshot
            )
        )


        print(
            f"Next training block opponent: "
            f"{opponent_name}\n"
        )


    # ========================================================
    # GOOGLE DRIVE CHECKPOINT — EVERY 25 GAMES
    #
    # This OVERWRITES latest.pt.
    # No new historical file is created.
    # ========================================================

    if (
        (game + 1)
        % SAVE_EVERY_GAMES
        == 0
    ):


        save_training_checkpoint(

            LATEST_CHECKPOINT,

            game=game,

            global_step=
                global_step,

            spatial_net=
                spatial_net,

            planners=
                planners,

            grand_planner=
                grand_planner,

            worker=
                worker,

            target_worker=
                target_worker,

            spatial_opt=
                spatial_opt,

            planner_opt=
                planner_opt,

            grand_opt=
                grand_opt,

            worker_opt=
                worker_opt,

            score_history=
                score_history,

            champion_game=
                champion_game
        )


        print(
            f"Saved latest training weights "
            f"to Google Drive at Game {game+1}."
        )

Kaggriculture horizon: 30 days x 24 turns/day = 720 turns/game
Expansion floor: at least 2 hands/day; force up to 3 total regions through day 18.
Mounted at /content/drive


/tmp/ipykernel_12730/609141274.py:3949: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(



TRAINING CHECKPOINT RESTORED
Starting from Game 23976
Global step: 17,238,025


CHAMPION LOADED: Game 23200
Weights are frozen during self-play.
Opponent adaptation remains enabled.


Validating loaded champion 23200 before self-play...


/tmp/ipykernel_12730/609141274.py:3949: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


Streaming output truncated to the last 5000 lines.
Game 38861 | Day  29/29 | Turn  696/720 | Phase 3 | Best WHEAT | CNN 0.85 | vs Champion 37800

GAME 38861 COMPLETE | FULL HIERARCHY
Opponent:            Champion 37800

Our score:           $12,217
Opponent score:      $16,877
Margin:              -4,660
Game reward:         -7.315

20-game avg margin:  -4,098
20-game win rate:    10%

Max money:           $12,217
Max strategic value: $30,457

Epsilon:             0.179
Exploration mode:    REHEAT #194
Cycle game:          61/200
CNN target weight:   0.850
Replay size:         150,000

Spatial CNN loss:    3.6694
Worker loss:         0.0381
Planner loss:        6.3100
Grand loss:          45.9950

ACTIONS
Plant:   168
Water:   414
Harvest: 76
Place:   30
Dig:     37

MARKET
BuySeed: 42
Sell:    41
Hire:    61
BuyLand: 2

Game 38862 | Day   0/29 | Turn    0/720 | Phase 3 | Best MELON | CNN 0.85 | vs Champion 37800 | SWITCH 0→3
Game 38862 | Day   1/29 | Turn   24/720 | Phase 3 | Best MEL